In [29]:
# EDA 09 — Style and Tempo Delta Analysis
# Cell 1 — Imports, connection, constants, and helpers

import os
import json
import math
import warnings
from decimal import Decimal

import numpy as np
import pandas as pd
import psycopg2

from scipy import stats
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# Locked season policy
# -------------------------------------------------------------------

DEV_SEASONS = [2022, 2023, 2024]
HOLDOUT_SEASON = 2025

# -------------------------------------------------------------------
# DB connection
# -------------------------------------------------------------------

conn = psycopg2.connect(
    host="127.0.0.1",
    port=5455,
    dbname="postgres",
    user="postgres",
    password="postgres"
)

cur = conn.cursor()

# -------------------------------------------------------------------
# Canonical tier assignment — do not modify
# -------------------------------------------------------------------

P4_CONFERENCES = {"ACC", "Big 12", "Big Ten", "SEC"}

def assign_tier(row):
    if row["team_name"] == "Notre Dame":
        return "P4"
    if row["team_name"] == "UConn":
        return "G5"
    if row["conference"] in P4_CONFERENCES:
        return "P4"
    return "G5"

# -------------------------------------------------------------------
# General helpers
# -------------------------------------------------------------------

def fetch_df(sql, params=None):
    """
    Execute SQL with psycopg2 and return a DataFrame.
    Keeps the shared connection open for the full notebook.
    """
    cur.execute(sql, params)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)


def cast_decimal_columns(df):
    """
    psycopg2 returns numeric columns as Decimal.
    Cast all Decimal-bearing columns to float64.
    """
    out = df.copy()
    for col in out.columns:
        if out[col].map(lambda x: isinstance(x, Decimal)).any():
            out[col] = out[col].astype(float)
    return out


def inspect_table(schema_name, table_name):
    """
    Inspect columns before writing SQL that depends on them.
    """
    sql = """
        SELECT
            table_schema,
            table_name,
            column_name,
            data_type,
            ordinal_position
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name = %s
        ORDER BY ordinal_position;
    """
    return fetch_df(sql, (schema_name, table_name))


# -------------------------------------------------------------------
# Schema introspection — required before SQL development
# -------------------------------------------------------------------

tables_to_inspect = [
    ("int", "int_game_team_features"),
    ("int", "int_team_season_features"),
    ("raw", "games"),
    ("raw", "plays"),
]

schema_outputs = {}

for schema_name, table_name in tables_to_inspect:
    df_schema = inspect_table(schema_name, table_name)
    schema_outputs[f"{schema_name}.{table_name}"] = df_schema
    
    print(f"\n--- {schema_name}.{table_name} ---")
    print(df_schema[["column_name", "data_type"]].to_string(index=False))

print("\nCell 1 complete.")
print("Confirm the schema output before running Cell 2.")


--- int.int_game_team_features ---
                column_name data_type
                    game_id    bigint
                     season   integer
                       week   integer
                  game_date      date
                  team_name      text
                   opponent      text
              points_scored   integer
             points_allowed   integer
                        win   integer
           off_epa_per_play   numeric
   def_epa_per_play_allowed   numeric
    close_game_epa_per_play   numeric
      close_game_play_count    bigint
close_game_def_epa_per_play   numeric
  close_game_def_play_count    bigint
                game_script      text
     game_script_avg_margin   numeric
          last3_off_epa_avg   numeric
              last3_win_pct   numeric
    last3_points_scored_avg   numeric
          last3_def_epa_avg   numeric
   last3_points_allowed_avg   numeric
       days_since_last_game   integer
 opp_sp_rating_at_game_time   numeric
              

In [30]:
# Cell 2 — Build valid FBS conference game pool

valid_games_sql = """
    DROP TABLE IF EXISTS temp_valid_fbs_conf_games;

    CREATE TEMP TABLE temp_valid_fbs_conf_games AS
    SELECT
        g.id AS game_id,
        g.season,
        g.week,
        g.start_date,
        g.neutral_site,
        g.conference_game,
        g.home_team,
        g.away_team,
        g.home_points,
        g.away_points,
        home_sf.conference AS home_conference,
        away_sf.conference AS away_conference,
        home_sf.sp_rating AS home_sp_rating,
        away_sf.sp_rating AS away_sp_rating
    FROM raw.games g
    INNER JOIN int.int_team_season_features home_sf
        ON g.home_team = home_sf.team_name
       AND g.season = home_sf.season
    INNER JOIN int.int_team_season_features away_sf
        ON g.away_team = away_sf.team_name
       AND g.season = away_sf.season
    WHERE g.season IN (2022, 2023, 2024)
      AND g.conference_game = TRUE
      AND home_sf.conference != 'FBS Independents'
      AND away_sf.conference != 'FBS Independents';

    ALTER TABLE temp_valid_fbs_conf_games
    ADD PRIMARY KEY (game_id);
"""

cur.execute(valid_games_sql)
conn.commit()

valid_games = cast_decimal_columns(fetch_df("""
    SELECT *
    FROM temp_valid_fbs_conf_games
    ORDER BY season, week, game_id;
"""))

valid_games["point_differential"] = valid_games["home_points"] - valid_games["away_points"]
valid_games["total_points"] = valid_games["home_points"] + valid_games["away_points"]
valid_games["sp_rating_delta"] = valid_games["home_sp_rating"] - valid_games["away_sp_rating"]

print("Valid FBS conference games:", len(valid_games))
print("Seasons:", sorted(valid_games["season"].unique().tolist()))
print("\nGame counts by season:")
print(valid_games["season"].value_counts().sort_index().to_string())

print("\nHome conference distribution:")
print(valid_games["home_conference"].value_counts().sort_index().to_string())

print("\nAway conference distribution:")
print(valid_games["away_conference"].value_counts().sort_index().to_string())

assert HOLDOUT_SEASON not in valid_games["season"].unique(), "ERROR: 2025 holdout leaked into valid_games."
assert valid_games["game_id"].isna().sum() == 0, "ERROR: Null game_id found."
assert valid_games["home_conference"].isna().sum() == 0, "ERROR: Null home_conference found."
assert valid_games["away_conference"].isna().sum() == 0, "ERROR: Null away_conference found."
assert "FBS Independents" not in set(valid_games["home_conference"]), "ERROR: FBS Independents found in home_conference."
assert "FBS Independents" not in set(valid_games["away_conference"]), "ERROR: FBS Independents found in away_conference."

print("\nCell 2 complete.")
print("FBS integrity check passed.")
print("Do not run Cell 3 until you confirm this output.")

Valid FBS conference games: 1607
Seasons: [2022, 2023, 2024]

Game counts by season:
season
2022    523
2023    546
2024    538

Home conference distribution:
home_conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       123
Mid-American         147
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Away conference distribution:
away_conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       123
Mid-American         147
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Cell 2 complete.
FBS integrity check passed.
Do not run Cell 3 until you confirm this output.


In [31]:
# Cell 3 — Aggregate raw.plays into one row per team per game style metrics
# FULL REWRITE: do not globally filter ppa IS NOT NULL.
# EPA metrics use ppa where available; non-EPA style metrics still use all scrimmage plays.

import time

start_time = time.time()

style_sql = """
    DROP TABLE IF EXISTS temp_team_game_style_metrics;

    CREATE TEMP TABLE temp_team_game_style_metrics AS
    WITH scrimmage_plays AS (
        SELECT
            p.game_id,
            p.drive_id,
            p.season,
            p.week,
            p.offense AS team_name,
            p.defense AS opponent,
            p.play_type,
            p.down,
            p.distance,
            p.yards_gained,
            p.yards_to_goal,
            p.ppa,
            p.scoring,
            p.offense_score,

            CASE
                WHEN p.play_type IN ('Rush', 'Rushing Touchdown') THEN 1
                ELSE 0
            END AS is_rush,

            CASE
                WHEN p.play_type IN (
                    'Pass Reception',
                    'Pass Incompletion',
                    'Passing Touchdown',
                    'Sack',
                    'Pass Completion',
                    'Pass Interception Return'
                ) THEN 1
                ELSE 0
            END AS is_pass,

            CASE
                WHEN p.down = 1
                  OR (p.down = 2 AND p.distance <= 8)
                  OR (p.down IN (3, 4) AND p.distance <= 5)
                THEN 1
                ELSE 0
            END AS is_std_down,

            CASE
                WHEN (p.down = 2 AND p.distance > 8)
                  OR (p.down IN (3, 4) AND p.distance > 5)
                THEN 1
                ELSE 0
            END AS is_pass_down,

            CASE
                WHEN p.distance IS NOT NULL
                 AND p.yards_gained IS NOT NULL
                 AND (
                    (p.down = 1 AND p.yards_gained >= 0.50 * p.distance)
                    OR (p.down = 2 AND p.yards_gained >= 0.70 * p.distance)
                    OR (p.down IN (3, 4) AND p.yards_gained >= p.distance)
                 )
                THEN 1
                ELSE 0
            END AS is_success,

            CASE
                WHEN p.yards_gained >= 10 THEN 1
                ELSE 0
            END AS is_explosive_10,

            CASE
                WHEN p.yards_gained >= 20 THEN 1
                ELSE 0
            END AS is_explosive_20,

            CASE
                WHEN p.play_type IN ('Rush', 'Rushing Touchdown')
                 AND p.yards_gained <= 0
                THEN 1
                ELSE 0
            END AS is_stuffed_rush,

            CASE
                WHEN p.play_type = 'Sack' THEN 1
                ELSE 0
            END AS is_sack,

            CASE
                WHEN p.yards_to_goal <= 40 THEN 1
                ELSE 0
            END AS is_scoring_opportunity,

            CASE
                WHEN p.play_type IN ('Rush', 'Rushing Touchdown') THEN
                    CASE
                        WHEN p.yards_gained IS NULL THEN NULL
                        WHEN p.yards_gained < 0 THEN 0
                        WHEN p.yards_gained <= 4 THEN p.yards_gained
                        ELSE 4
                    END
                ELSE NULL
            END AS line_yards
        FROM raw.plays p
        INNER JOIN temp_valid_fbs_conf_games vg
            ON p.game_id = vg.game_id
        WHERE p.season IN (2022, 2023, 2024)
          AND p.play_type IN (
              'Rush',
              'Rushing Touchdown',
              'Pass Reception',
              'Pass Incompletion',
              'Passing Touchdown',
              'Sack',
              'Pass Completion',
              'Pass Interception Return'
          )
    ),

    offense_metrics AS (
        SELECT
            game_id,
            season,
            week,
            team_name,
            opponent,

            COUNT(*) AS off_scrimmage_plays,

            AVG(CASE WHEN is_rush = 1 THEN is_success::numeric END) AS off_success_rate_rush,
            AVG(CASE WHEN is_pass = 1 THEN is_success::numeric END) AS off_success_rate_pass,
            AVG(CASE WHEN is_std_down = 1 THEN is_success::numeric END) AS off_success_rate_std_downs,
            AVG(CASE WHEN is_pass_down = 1 THEN is_success::numeric END) AS off_success_rate_pass_downs,

            AVG(CASE WHEN is_rush = 1 THEN ppa END) AS off_epa_rush,
            AVG(CASE WHEN is_pass = 1 THEN ppa END) AS off_epa_pass,

            AVG(is_explosive_10::numeric) AS off_explosive_rate_10,
            AVG(is_explosive_20::numeric) AS off_explosive_rate_20,

            AVG(CASE WHEN is_rush = 1 THEN is_stuffed_rush::numeric END) AS off_stuff_rate,
            AVG(line_yards) AS off_line_yards_per_rush,

            AVG(CASE WHEN is_std_down = 1 THEN is_rush::numeric END) AS rush_rate_std_downs,
            AVG(CASE WHEN is_pass_down = 1 THEN is_rush::numeric END) AS rush_rate_pass_downs,

            AVG(CASE WHEN is_pass = 1 THEN is_sack::numeric END) AS off_sack_rate_allowed
        FROM scrimmage_plays
        GROUP BY
            game_id,
            season,
            week,
            team_name,
            opponent
    ),

    scoring_opp_drive_points AS (
        SELECT
            game_id,
            team_name,
            drive_id,
            MAX(offense_score) - MIN(offense_score) AS scoring_opp_points
        FROM scrimmage_plays
        WHERE is_scoring_opportunity = 1
        GROUP BY
            game_id,
            team_name,
            drive_id
    ),

    points_per_opp AS (
        SELECT
            game_id,
            team_name,
            AVG(scoring_opp_points::numeric) AS off_pts_per_opportunity
        FROM scoring_opp_drive_points
        GROUP BY
            game_id,
            team_name
    ),

    offense_final AS (
        SELECT
            om.*,
            ppo.off_pts_per_opportunity
        FROM offense_metrics om
        LEFT JOIN points_per_opp ppo
            ON om.game_id = ppo.game_id
           AND om.team_name = ppo.team_name
    ),

    defense_base AS (
        SELECT
            game_id,
            season,
            week,
            opponent AS team_name,
            team_name AS opponent,

            COUNT(*) AS def_scrimmage_plays,

            AVG(CASE WHEN is_rush = 1 THEN is_success::numeric END) AS def_success_rate_rush_allowed,
            AVG(CASE WHEN is_pass = 1 THEN is_success::numeric END) AS def_success_rate_pass_allowed,
            AVG(CASE WHEN is_std_down = 1 THEN is_success::numeric END) AS def_success_rate_std_downs_allowed,

            AVG(CASE WHEN is_rush = 1 THEN ppa END) AS def_epa_rush_allowed,
            AVG(CASE WHEN is_pass = 1 THEN ppa END) AS def_epa_pass_allowed,

            AVG(CASE WHEN is_rush = 1 THEN is_stuffed_rush::numeric END) AS def_stuff_rate_allowed,
            AVG(is_explosive_10::numeric) AS def_explosive_rate_10_allowed,
            AVG(is_explosive_20::numeric) AS def_explosive_rate_20_allowed,

            AVG(CASE WHEN is_pass = 1 THEN is_sack::numeric END) AS def_sack_rate
        FROM scrimmage_plays
        GROUP BY
            game_id,
            season,
            week,
            opponent,
            team_name
    ),

    def_points_per_opp AS (
        SELECT
            game_id,
            opponent AS team_name,
            AVG(scoring_opp_points::numeric) AS def_pts_per_opportunity_allowed
        FROM (
            SELECT
                sp.game_id,
                sp.team_name,
                sp.opponent,
                sp.drive_id,
                MAX(sp.offense_score) - MIN(sp.offense_score) AS scoring_opp_points
            FROM scrimmage_plays sp
            WHERE sp.is_scoring_opportunity = 1
            GROUP BY
                sp.game_id,
                sp.team_name,
                sp.opponent,
                sp.drive_id
        ) x
        GROUP BY
            game_id,
            opponent
    ),

    defense_final AS (
        SELECT
            db.*,
            dppo.def_pts_per_opportunity_allowed
        FROM defense_base db
        LEFT JOIN def_points_per_opp dppo
            ON db.game_id = dppo.game_id
           AND db.team_name = dppo.team_name
    )

    SELECT
        o.game_id,
        o.season,
        o.week,
        o.team_name,
        o.opponent,

        o.off_scrimmage_plays,
        d.def_scrimmage_plays,

        o.off_success_rate_rush,
        o.off_success_rate_pass,
        o.off_success_rate_std_downs,
        o.off_success_rate_pass_downs,
        o.off_epa_rush,
        o.off_epa_pass,
        o.off_explosive_rate_10,
        o.off_explosive_rate_20,
        o.off_stuff_rate,
        o.off_line_yards_per_rush,
        o.off_pts_per_opportunity,
        o.rush_rate_std_downs,
        o.rush_rate_pass_downs,
        o.off_sack_rate_allowed,

        d.def_success_rate_rush_allowed,
        d.def_success_rate_pass_allowed,
        d.def_success_rate_std_downs_allowed,
        d.def_epa_rush_allowed,
        d.def_epa_pass_allowed,
        d.def_stuff_rate_allowed,
        d.def_explosive_rate_10_allowed,
        d.def_explosive_rate_20_allowed,
        d.def_pts_per_opportunity_allowed,
        d.def_sack_rate
    FROM offense_final o
    INNER JOIN defense_final d
        ON o.game_id = d.game_id
       AND o.team_name = d.team_name
       AND o.opponent = d.opponent;

    ALTER TABLE temp_team_game_style_metrics
    ADD PRIMARY KEY (game_id, team_name);
"""

cur.execute(style_sql)
conn.commit()

elapsed = time.time() - start_time

team_game_style = cast_decimal_columns(fetch_df("""
    SELECT *
    FROM temp_team_game_style_metrics
    ORDER BY season, week, game_id, team_name;
"""))

print(f"raw.plays aggregation elapsed seconds: {elapsed:.2f}")
print("Team-game style rows:", len(team_game_style))
print("Unique games:", team_game_style["game_id"].nunique())
print("Expected rows if exactly two teams per game:", valid_games["game_id"].nunique() * 2)

print("\nRows by season:")
print(team_game_style["season"].value_counts().sort_index().to_string())

print("\nNull counts:")
style_cols = [
    "off_success_rate_rush",
    "off_success_rate_pass",
    "off_success_rate_std_downs",
    "off_success_rate_pass_downs",
    "off_epa_rush",
    "off_epa_pass",
    "off_explosive_rate_10",
    "off_explosive_rate_20",
    "off_stuff_rate",
    "off_line_yards_per_rush",
    "off_pts_per_opportunity",
    "rush_rate_std_downs",
    "rush_rate_pass_downs",
    "off_sack_rate_allowed",
    "def_success_rate_rush_allowed",
    "def_success_rate_pass_allowed",
    "def_success_rate_std_downs_allowed",
    "def_epa_rush_allowed",
    "def_epa_pass_allowed",
    "def_stuff_rate_allowed",
    "def_explosive_rate_10_allowed",
    "def_explosive_rate_20_allowed",
    "def_pts_per_opportunity_allowed",
    "def_sack_rate",
]
print(team_game_style[style_cols].isna().sum().sort_values(ascending=False).to_string())

assert elapsed < 30, f"ERROR: raw.plays aggregation took {elapsed:.2f}s, exceeding 30s requirement."
assert HOLDOUT_SEASON not in team_game_style["season"].unique(), "ERROR: 2025 holdout leaked into style metrics."
assert team_game_style["game_id"].nunique() == valid_games["game_id"].nunique(), "ERROR: Game count mismatch."
assert len(team_game_style) == valid_games["game_id"].nunique() * 2, "ERROR: Expected exactly two team rows per game."

print("\nCell 3 complete.")
print("Do not run Cell 4 until you confirm this output.")

raw.plays aggregation elapsed seconds: 2.09
Team-game style rows: 3214
Unique games: 1607
Expected rows if exactly two teams per game: 3214

Rows by season:
season
2022    1046
2023    1092
2024    1076

Null counts:
def_pts_per_opportunity_allowed       7
off_pts_per_opportunity               7
off_epa_rush                          6
off_epa_pass                          6
def_epa_pass_allowed                  6
def_epa_rush_allowed                  6
off_success_rate_rush                 0
def_success_rate_rush_allowed         0
def_explosive_rate_20_allowed         0
def_explosive_rate_10_allowed         0
def_stuff_rate_allowed                0
def_success_rate_std_downs_allowed    0
def_success_rate_pass_allowed         0
rush_rate_pass_downs                  0
off_sack_rate_allowed                 0
off_success_rate_pass                 0
rush_rate_std_downs                   0
off_line_yards_per_rush               0
off_stuff_rate                        0
off_explosive_rate_20  

In [32]:
# Cell 4 — Build home-minus-away style delta table with outcomes and controls

home_style = team_game_style.merge(
    valid_games[
        [
            "game_id",
            "season",
            "week",
            "home_team",
            "away_team",
            "home_conference",
            "away_conference",
            "home_points",
            "away_points",
            "point_differential",
            "total_points",
            "sp_rating_delta",
        ]
    ],
    on=["game_id", "season", "week"],
    how="inner",
)

home_style = home_style[home_style["team_name"] == home_style["home_team"]].copy()

away_style = team_game_style.merge(
    valid_games[
        [
            "game_id",
            "season",
            "week",
            "home_team",
            "away_team",
            "home_conference",
            "away_conference",
            "home_points",
            "away_points",
            "point_differential",
            "total_points",
            "sp_rating_delta",
        ]
    ],
    on=["game_id", "season", "week"],
    how="inner",
)

away_style = away_style[away_style["team_name"] == away_style["away_team"]].copy()

print("Home rows:", len(home_style))
print("Away rows:", len(away_style))

assert len(home_style) == len(valid_games), "ERROR: Home style rows do not match valid game count."
assert len(away_style) == len(valid_games), "ERROR: Away style rows do not match valid game count."

delta_base_cols = [
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "home_conference",
    "away_conference",
    "home_points",
    "away_points",
    "point_differential",
    "total_points",
    "sp_rating_delta",
]

style_metric_cols = [
    "off_success_rate_rush",
    "off_success_rate_pass",
    "off_success_rate_std_downs",
    "off_success_rate_pass_downs",
    "off_epa_rush",
    "off_epa_pass",
    "off_explosive_rate_10",
    "off_explosive_rate_20",
    "off_stuff_rate",
    "off_line_yards_per_rush",
    "off_pts_per_opportunity",
    "rush_rate_std_downs",
    "rush_rate_pass_downs",
    "off_sack_rate_allowed",
    "def_success_rate_rush_allowed",
    "def_success_rate_pass_allowed",
    "def_success_rate_std_downs_allowed",
    "def_epa_rush_allowed",
    "def_epa_pass_allowed",
    "def_stuff_rate_allowed",
    "def_explosive_rate_10_allowed",
    "def_explosive_rate_20_allowed",
    "def_pts_per_opportunity_allowed",
    "def_sack_rate",
]

home_prefixed = home_style[delta_base_cols + style_metric_cols].copy()
away_prefixed = away_style[["game_id"] + style_metric_cols].copy()

home_prefixed = home_prefixed.rename(
    columns={col: f"home_{col}" for col in style_metric_cols}
)

away_prefixed = away_prefixed.rename(
    columns={col: f"away_{col}" for col in style_metric_cols}
)

style_delta = home_prefixed.merge(
    away_prefixed,
    on="game_id",
    how="inner",
)

for col in style_metric_cols:
    style_delta[f"{col}_delta"] = style_delta[f"home_{col}"] - style_delta[f"away_{col}"]

# Conference label for stratified testing.
# For conference games, home and away conference should match.
style_delta["conference"] = style_delta["home_conference"]

# Tier assignment for home team conference/tier.
style_delta["team_name"] = style_delta["home_team"]
style_delta["conference_for_tier"] = style_delta["home_conference"]
style_delta["conference"] = style_delta["home_conference"]

style_delta_for_tier = style_delta.rename(columns={"conference": "conference_original"})
style_delta_for_tier["conference"] = style_delta_for_tier["conference_for_tier"]
style_delta["tier"] = style_delta_for_tier.apply(assign_tier, axis=1)

# Control: close_game_epa_delta from int_game_team_features.
controls_sql = """
    SELECT
        f.game_id,
        f.season,
        f.week,
        f.team_name,
        f.opponent,
        f.close_game_epa_per_play,
        f.close_game_def_epa_per_play
    FROM int.int_game_team_features f
    INNER JOIN temp_valid_fbs_conf_games vg
        ON f.game_id = vg.game_id
    WHERE f.season IN (2022, 2023, 2024);
"""

controls_team = cast_decimal_columns(fetch_df(controls_sql))

home_controls = controls_team.merge(
    valid_games[["game_id", "home_team", "away_team"]],
    on="game_id",
    how="inner",
)
home_controls = home_controls[home_controls["team_name"] == home_controls["home_team"]].copy()

away_controls = controls_team.merge(
    valid_games[["game_id", "home_team", "away_team"]],
    on="game_id",
    how="inner",
)
away_controls = away_controls[away_controls["team_name"] == away_controls["away_team"]].copy()

home_controls = home_controls[
    [
        "game_id",
        "close_game_epa_per_play",
        "close_game_def_epa_per_play",
    ]
].rename(
    columns={
        "close_game_epa_per_play": "home_close_game_epa_per_play",
        "close_game_def_epa_per_play": "home_close_game_def_epa_per_play",
    }
)

away_controls = away_controls[
    [
        "game_id",
        "close_game_epa_per_play",
        "close_game_def_epa_per_play",
    ]
].rename(
    columns={
        "close_game_epa_per_play": "away_close_game_epa_per_play",
        "close_game_def_epa_per_play": "away_close_game_def_epa_per_play",
    }
)

style_delta = style_delta.merge(home_controls, on="game_id", how="left")
style_delta = style_delta.merge(away_controls, on="game_id", how="left")

style_delta["close_game_epa_delta"] = (
    style_delta["home_close_game_epa_per_play"]
    - style_delta["away_close_game_epa_per_play"]
)

style_delta["close_game_def_epa_delta"] = (
    style_delta["home_close_game_def_epa_per_play"]
    - style_delta["away_close_game_def_epa_per_play"]
)

# Primary control requested by session state:
# close_game_epa_delta and sp_rating_delta.
# Keep close_game_def_epa_delta available for diagnostics but do not use in the primary partial-r test.

delta_cols = [f"{col}_delta" for col in style_metric_cols]

print("Style delta rows:", len(style_delta))
print("Unique games:", style_delta["game_id"].nunique())

print("\nSeasons:")
print(style_delta["season"].value_counts().sort_index().to_string())

print("\nConference distribution:")
print(style_delta["conference"].value_counts().sort_index().to_string())

print("\nTier distribution:")
print(style_delta["tier"].value_counts().sort_index().to_string())

print("\nRequired control null counts:")
required_control_cols = [
    "point_differential",
    "total_points",
    "sp_rating_delta",
    "close_game_epa_delta",
]
print(style_delta[required_control_cols].isna().sum().to_string())

print("\nDelta null counts:")
print(style_delta[delta_cols].isna().sum().sort_values(ascending=False).to_string())

assert len(style_delta) == len(valid_games), "ERROR: style_delta row count does not match valid_games."
assert style_delta["game_id"].nunique() == len(valid_games), "ERROR: style_delta unique game count mismatch."
assert HOLDOUT_SEASON not in style_delta["season"].unique(), "ERROR: 2025 holdout leaked into style_delta."
assert "FBS Independents" not in set(style_delta["conference"]), "ERROR: FBS Independents found in style_delta."
required_control_nulls = style_delta[required_control_cols].isna().sum().sum()

if required_control_nulls > 0:
    print(
        f"\nWARNING: {required_control_nulls} required-control null values found. "
        "Do not use style_delta directly for signal testing. "
        "Run Cell 4A to create style_delta_analysis."
    )
else:
    print("\nRequired-control check passed.")

print("\nCell 4 complete.")
print("Do not run Cell 5 until you confirm this output.")

Home rows: 1607
Away rows: 1607
Style delta rows: 1607
Unique games: 1607

Seasons:
season
2022    523
2023    546
2024    538

Conference distribution:
conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       123
Mid-American         147
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Tier distribution:
tier
G5    853
P4    754

Required control null counts:
point_differential      0
total_points            0
sp_rating_delta         0
close_game_epa_delta    3

Delta null counts:
def_pts_per_opportunity_allowed_delta       7
off_pts_per_opportunity_delta               7
off_epa_rush_delta                          3
off_epa_pass_delta                          3
def_epa_pass_allowed_delta                  3
def_epa_rush_allowed_delta                  3
off_success_rate_rush_delta                 0
def_success_rate_rush_allowed_delta         0
def_explosive_rate

In [33]:
# Cell 4A — Create analysis table excluding games missing required primary controls

primary_required_cols = [
    "point_differential",
    "total_points",
    "sp_rating_delta",
    "close_game_epa_delta",
]

missing_primary_controls = style_delta[style_delta[primary_required_cols].isna().any(axis=1)].copy()

print("Rows missing required primary controls:", len(missing_primary_controls))

if len(missing_primary_controls) > 0:
    display(
        missing_primary_controls[
            [
                "game_id",
                "season",
                "week",
                "home_team",
                "away_team",
                "home_conference",
                "away_conference",
                "point_differential",
                "total_points",
                "sp_rating_delta",
                "close_game_epa_delta",
            ]
        ].sort_values(["season", "week", "game_id"])
    )

style_delta_analysis = style_delta.dropna(subset=primary_required_cols).copy()

print("\nBase style_delta rows:", len(style_delta))
print("Analysis rows after required-control drop:", len(style_delta_analysis))
print("Dropped rows:", len(style_delta) - len(style_delta_analysis))

print("\nAnalysis seasons:")
print(style_delta_analysis["season"].value_counts().sort_index().to_string())

print("\nAnalysis conference distribution:")
print(style_delta_analysis["conference"].value_counts().sort_index().to_string())

print("\nAnalysis tier distribution:")
print(style_delta_analysis["tier"].value_counts().sort_index().to_string())

print("\nRemaining required-control null counts:")
print(style_delta_analysis[primary_required_cols].isna().sum().to_string())

assert len(style_delta_analysis) == 1604, "ERROR: Expected 1604 analysis games after dropping 3 missing-control games."
assert style_delta_analysis[primary_required_cols].isna().sum().sum() == 0, "ERROR: Required control nulls remain."
assert HOLDOUT_SEASON not in style_delta_analysis["season"].unique(), "ERROR: 2025 holdout leaked into style_delta_analysis."
assert "FBS Independents" not in set(style_delta_analysis["conference"]), "ERROR: FBS Independents found."

print("\nCell 4A complete.")
print("Use style_delta_analysis for partial-r signal testing.")
print("Do not run Cell 5 until you confirm this output.")

Rows missing required primary controls: 3


,game_id,season,week,home_team,away_team,home_conference,away_conference,point_differential,total_points,sp_rating_delta,close_game_epa_delta
1104,401641034,2024,4,Sam Houston,New Mexico State,Conference USA,Conference USA,20,42,12.1,NaN
1471,401644689,2024,12,Miami (OH),Kent State,Mid-American,Mid-American,27,41,34.3,NaN
1589,401644780,2024,14,Western Michigan,Eastern Michigan,Mid-American,Mid-American,8,44,3.1,NaN



Base style_delta rows: 1607
Analysis rows after required-control drop: 1604
Dropped rows: 3

Analysis seasons:
season
2022    523
2023    546
2024    535

Analysis conference distribution:
conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       122
Mid-American         145
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Analysis tier distribution:
tier
G5    850
P4    754

Remaining required-control null counts:
point_differential      0
total_points            0
sp_rating_delta         0
close_game_epa_delta    0

Cell 4A complete.
Use style_delta_analysis for partial-r signal testing.
Do not run Cell 5 until you confirm this output.


In [34]:
# Cell 5 — Partial-r helper functions and primary spread / total-points signal tests

SIGNAL_THRESHOLD = 0.08

PRIMARY_CONTROLS = [
    "close_game_epa_delta",
    "sp_rating_delta",
]

STYLE_DELTA_COLS = [f"{col}_delta" for col in style_metric_cols]

def partial_corr(df, x_col, y_col, control_cols):
    """
    Compute partial correlation between x and y after residualizing both
    against control_cols.
    """
    needed_cols = [x_col, y_col] + control_cols
    d = df[needed_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    
    n = len(d)
    k = len(control_cols)
    
    if n <= k + 3:
        return {
            "n": n,
            "partial_r": np.nan,
            "p_value": np.nan,
        }
    
    x = d[[x_col]].values
    y = d[y_col].values
    controls = d[control_cols].values
    
    x_model = LinearRegression()
    y_model = LinearRegression()
    
    x_resid = x[:, 0] - x_model.fit(controls, x[:, 0]).predict(controls)
    y_resid = y - y_model.fit(controls, y).predict(controls)
    
    if np.std(x_resid) == 0 or np.std(y_resid) == 0:
        return {
            "n": n,
            "partial_r": np.nan,
            "p_value": np.nan,
        }
    
    r, p = stats.pearsonr(x_resid, y_resid)
    
    return {
        "n": n,
        "partial_r": r,
        "p_value": p,
    }


def make_strata(df):
    """
    Return ordered stratification dictionary:
    full population, P4, G5, then individual conferences.
    """
    strata = {
        "Full": df.copy(),
        "P4": df[df["tier"] == "P4"].copy(),
        "G5": df[df["tier"] == "G5"].copy(),
    }
    
    for conf in sorted(df["conference"].dropna().unique()):
        strata[conf] = df[df["conference"] == conf].copy()
    
    return strata


def run_signal_tests(df, feature_cols, outcome_col, outcome_label, control_cols):
    """
    Run partial-r tests for each feature delta across all required strata.
    """
    rows = []
    strata = make_strata(df)
    
    for stratum_name, stratum_df in strata.items():
        for feature_col in feature_cols:
            result = partial_corr(
                stratum_df,
                x_col=feature_col,
                y_col=outcome_col,
                control_cols=control_cols,
            )
            
            partial_r = result["partial_r"]
            
            rows.append({
                "outcome": outcome_label,
                "outcome_col": outcome_col,
                "feature_delta": feature_col,
                "stratum": stratum_name,
                "n": result["n"],
                "partial_r": partial_r,
                "abs_partial_r": abs(partial_r) if pd.notna(partial_r) else np.nan,
                "p_value": result["p_value"],
                "threshold": SIGNAL_THRESHOLD,
                "clears_threshold": (
                    abs(partial_r) >= SIGNAL_THRESHOLD
                    if pd.notna(partial_r)
                    else False
                ),
                "direction": (
                    "positive"
                    if pd.notna(partial_r) and partial_r > 0
                    else "negative"
                    if pd.notna(partial_r) and partial_r < 0
                    else "none"
                ),
            })
    
    return pd.DataFrame(rows)


spread_signal = run_signal_tests(
    df=style_delta_analysis,
    feature_cols=STYLE_DELTA_COLS,
    outcome_col="point_differential",
    outcome_label="spread",
    control_cols=PRIMARY_CONTROLS,
)

total_signal = run_signal_tests(
    df=style_delta_analysis,
    feature_cols=STYLE_DELTA_COLS,
    outcome_col="total_points",
    outcome_label="over_under",
    control_cols=PRIMARY_CONTROLS,
)

primary_signal = pd.concat([spread_signal, total_signal], ignore_index=True)

print("Primary signal rows:", len(primary_signal))
print("Outcomes:")
print(primary_signal["outcome"].value_counts().to_string())

print("\nStrata:")
print(primary_signal["stratum"].value_counts().sort_index().to_string())

print("\nSpread — full population, sorted by abs partial r:")
display(
    primary_signal[
        (primary_signal["outcome"] == "spread")
        & (primary_signal["stratum"] == "Full")
    ]
    .sort_values("abs_partial_r", ascending=False)
    .reset_index(drop=True)
)

print("\nOver/under — full population, sorted by abs partial r:")
display(
    primary_signal[
        (primary_signal["outcome"] == "over_under")
        & (primary_signal["stratum"] == "Full")
    ]
    .sort_values("abs_partial_r", ascending=False)
    .reset_index(drop=True)
)

print("\nThreshold clear counts by outcome and stratum:")
display(
    primary_signal
    .groupby(["outcome", "stratum"], as_index=False)["clears_threshold"]
    .sum()
    .sort_values(["outcome", "stratum"])
)

assert set(primary_signal["outcome"]) == {"spread", "over_under"}, "ERROR: Missing primary outcomes."
assert "Full" in set(primary_signal["stratum"]), "ERROR: Full stratum missing."
assert "P4" in set(primary_signal["stratum"]), "ERROR: P4 stratum missing."
assert "G5" in set(primary_signal["stratum"]), "ERROR: G5 stratum missing."

print("\nCell 5 complete.")
print("Share the full-population tables and threshold counts before Cell 6.")

Primary signal rows: 624
Outcomes:
outcome
spread        312
over_under    312

Strata:
stratum
ACC                  48
American Athletic    48
Big 12               48
Big Ten              48
Conference USA       48
Full                 48
G5                   48
Mid-American         48
Mountain West        48
P4                   48
Pac-12               48
SEC                  48
Sun Belt             48

Spread — full population, sorted by abs partial r:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction
0,spread,point_differential,rush_rate_std_downs_delta,Full,1604,0.292691,0.292691,4.725394e-33,0.08,True,positive
1,spread,point_differential,rush_rate_pass_downs_delta,Full,1604,0.216574,0.216574,1.765773e-18,0.08,True,positive
2,spread,point_differential,off_pts_per_opportunity_delta,Full,1597,0.146895,0.146895,3.686077e-09,0.08,True,positive
3,spread,point_differential,def_pts_per_opportunity_allowed_delta,Full,1597,-0.146895,0.146895,3.686077e-09,0.08,True,negative
4,spread,point_differential,def_success_rate_pass_allowed_delta,Full,1604,-0.142680,0.142680,9.509699e-09,0.08,True,negative
5,spread,point_differential,off_success_rate_pass_delta,Full,1604,0.142680,0.142680,9.509699e-09,0.08,True,positive
6,spread,point_differential,def_epa_pass_allowed_delta,Full,1604,-0.088056,0.088056,4.143906e-04,0.08,True,negative
7,spread,point_differential,off_epa_pass_delta,Full,1604,0.088056,0.088056,4.143906e-04,0.08,True,positive
8,spread,point_differential,off_success_rate_pass_downs_delta,Full,1604,0.075526,0.075526,2.471749e-03,0.08,False,positive
9,spread,point_differential,off_sack_rate_allowed_delta,Full,1604,-0.071307,0.071307,4.273155e-03,0.08,False,negative



Over/under — full population, sorted by abs partial r:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction
0,over_under,total_points,off_success_rate_pass_downs_delta,Full,1604,0.044432,0.044432,0.075239,0.08,False,positive
1,over_under,total_points,off_epa_pass_delta,Full,1604,0.030080,0.030080,0.228569,0.08,False,positive
2,over_under,total_points,def_epa_pass_allowed_delta,Full,1604,-0.030080,0.030080,0.228569,0.08,False,negative
3,over_under,total_points,rush_rate_pass_downs_delta,Full,1604,-0.029148,0.029148,0.243323,0.08,False,negative
4,over_under,total_points,def_success_rate_pass_allowed_delta,Full,1604,-0.025254,0.025254,0.312113,0.08,False,negative
5,over_under,total_points,off_success_rate_pass_delta,Full,1604,0.025254,0.025254,0.312113,0.08,False,positive
6,over_under,total_points,off_explosive_rate_10_delta,Full,1604,-0.022831,0.022831,0.360827,0.08,False,negative
7,over_under,total_points,def_explosive_rate_10_allowed_delta,Full,1604,0.022831,0.022831,0.360827,0.08,False,positive
8,over_under,total_points,off_success_rate_std_downs_delta,Full,1604,-0.018308,0.018308,0.463726,0.08,False,negative
9,over_under,total_points,def_success_rate_std_downs_allowed_delta,Full,1604,0.018308,0.018308,0.463726,0.08,False,positive



Threshold clear counts by outcome and stratum:


,outcome,stratum,clears_threshold
0,over_under,ACC,9
1,over_under,American Athletic,2
2,over_under,Big 12,8
3,over_under,Big Ten,9
4,over_under,Conference USA,9
5,over_under,Full,0
6,over_under,G5,0
7,over_under,Mid-American,14
8,over_under,Mountain West,5
9,over_under,P4,1



Cell 5 complete.
Share the full-population tables and threshold counts before Cell 6.


In [35]:
# Cell 5B — Moneyline variance signal test
# Tests whether style deltas explain residual spread variance after controls.
# This is the EDA proxy for moneyline variance signal.

def add_spread_residual_variance_targets(df, outcome_col, control_cols):
    """
    Create residual variance targets from point differential after baseline controls.
    
    Moneyline depends on the probability mass around the expected margin.
    If a style delta explains residual spread volatility, it may affect moneyline variance.
    """
    out = df.copy()
    
    needed_cols = [outcome_col] + control_cols
    d = out[needed_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    
    baseline_model = LinearRegression()
    X = d[control_cols].values
    y = d[outcome_col].values
    
    baseline_model.fit(X, y)
    y_hat = baseline_model.predict(X)
    resid = y - y_hat
    
    out["spread_residual"] = np.nan
    out.loc[d.index, "spread_residual"] = resid
    
    out["abs_spread_residual"] = out["spread_residual"].abs()
    out["squared_spread_residual"] = out["spread_residual"] ** 2
    
    return out


style_delta_analysis = add_spread_residual_variance_targets(
    df=style_delta_analysis,
    outcome_col="point_differential",
    control_cols=PRIMARY_CONTROLS,
)

moneyline_abs_signal = run_signal_tests(
    df=style_delta_analysis,
    feature_cols=STYLE_DELTA_COLS,
    outcome_col="abs_spread_residual",
    outcome_label="moneyline_abs_margin_variance",
    control_cols=PRIMARY_CONTROLS,
)

moneyline_sq_signal = run_signal_tests(
    df=style_delta_analysis,
    feature_cols=STYLE_DELTA_COLS,
    outcome_col="squared_spread_residual",
    outcome_label="moneyline_squared_margin_variance",
    control_cols=PRIMARY_CONTROLS,
)

moneyline_signal = pd.concat(
    [moneyline_abs_signal, moneyline_sq_signal],
    ignore_index=True
)

print("Moneyline variance signal rows:", len(moneyline_signal))
print("Outcomes:")
print(moneyline_signal["outcome"].value_counts().to_string())

print("\nMoneyline abs residual variance — full population, sorted by abs partial r:")
display(
    moneyline_signal[
        (moneyline_signal["outcome"] == "moneyline_abs_margin_variance")
        & (moneyline_signal["stratum"] == "Full")
    ]
    .sort_values("abs_partial_r", ascending=False)
    .reset_index(drop=True)
)

print("\nMoneyline squared residual variance — full population, sorted by abs partial r:")
display(
    moneyline_signal[
        (moneyline_signal["outcome"] == "moneyline_squared_margin_variance")
        & (moneyline_signal["stratum"] == "Full")
    ]
    .sort_values("abs_partial_r", ascending=False)
    .reset_index(drop=True)
)

print("\nMoneyline threshold clear counts by variance target and stratum:")
display(
    moneyline_signal
    .groupby(["outcome", "stratum"], as_index=False)["clears_threshold"]
    .sum()
    .sort_values(["outcome", "stratum"])
)

# Combine primary spread/O-U tests with moneyline variance tests.
all_signal = pd.concat(
    [primary_signal, moneyline_signal],
    ignore_index=True
)

print("\nAll signal rows after adding moneyline:", len(all_signal))
print("All outcomes:")
print(all_signal["outcome"].value_counts().to_string())

assert "moneyline_abs_margin_variance" in set(all_signal["outcome"]), "ERROR: Missing moneyline abs variance test."
assert "moneyline_squared_margin_variance" in set(all_signal["outcome"]), "ERROR: Missing moneyline squared variance test."
assert "spread" in set(all_signal["outcome"]), "ERROR: Missing spread test."
assert "over_under" in set(all_signal["outcome"]), "ERROR: Missing over/under test."

print("\nCell 5B complete.")
print("Use all_signal, not primary_signal alone, for later verdict construction.")
print("Share the full-population moneyline tables and threshold counts before Cell 6.")

Moneyline variance signal rows: 624
Outcomes:
outcome
moneyline_abs_margin_variance        312
moneyline_squared_margin_variance    312

Moneyline abs residual variance — full population, sorted by abs partial r:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction
0,moneyline_abs_margin_variance,abs_spread_residual,def_sack_rate_delta,Full,1604,-0.091936,0.091936,0.000227,0.08,True,negative
1,moneyline_abs_margin_variance,abs_spread_residual,off_sack_rate_allowed_delta,Full,1604,0.091936,0.091936,0.000227,0.08,True,positive
2,moneyline_abs_margin_variance,abs_spread_residual,off_success_rate_pass_downs_delta,Full,1604,0.036823,0.036823,0.140453,0.08,False,positive
3,moneyline_abs_margin_variance,abs_spread_residual,def_success_rate_pass_allowed_delta,Full,1604,-0.036509,0.036509,0.143869,0.08,False,negative
4,moneyline_abs_margin_variance,abs_spread_residual,off_success_rate_pass_delta,Full,1604,0.036509,0.036509,0.143869,0.08,False,positive
5,moneyline_abs_margin_variance,abs_spread_residual,off_explosive_rate_20_delta,Full,1604,-0.035570,0.035570,0.154469,0.08,False,negative
6,moneyline_abs_margin_variance,abs_spread_residual,def_explosive_rate_20_allowed_delta,Full,1604,0.035570,0.035570,0.154469,0.08,False,positive
7,moneyline_abs_margin_variance,abs_spread_residual,off_stuff_rate_delta,Full,1604,-0.032796,0.032796,0.189243,0.08,False,negative
8,moneyline_abs_margin_variance,abs_spread_residual,def_stuff_rate_allowed_delta,Full,1604,0.032796,0.032796,0.189243,0.08,False,positive
9,moneyline_abs_margin_variance,abs_spread_residual,rush_rate_std_downs_delta,Full,1604,-0.031003,0.031003,0.214606,0.08,False,negative



Moneyline squared residual variance — full population, sorted by abs partial r:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction
0,moneyline_squared_margin_variance,squared_spread_residual,def_sack_rate_delta,Full,1604,-0.078488,0.078488,0.001656,0.08,False,negative
1,moneyline_squared_margin_variance,squared_spread_residual,off_sack_rate_allowed_delta,Full,1604,0.078488,0.078488,0.001656,0.08,False,positive
2,moneyline_squared_margin_variance,squared_spread_residual,def_explosive_rate_20_allowed_delta,Full,1604,0.040367,0.040367,0.106073,0.08,False,positive
3,moneyline_squared_margin_variance,squared_spread_residual,off_explosive_rate_20_delta,Full,1604,-0.040367,0.040367,0.106073,0.08,False,negative
4,moneyline_squared_margin_variance,squared_spread_residual,off_success_rate_pass_downs_delta,Full,1604,0.033429,0.033429,0.180839,0.08,False,positive
5,moneyline_squared_margin_variance,squared_spread_residual,def_explosive_rate_10_allowed_delta,Full,1604,-0.030091,0.030091,0.228401,0.08,False,negative
6,moneyline_squared_margin_variance,squared_spread_residual,off_explosive_rate_10_delta,Full,1604,0.030091,0.030091,0.228401,0.08,False,positive
7,moneyline_squared_margin_variance,squared_spread_residual,off_stuff_rate_delta,Full,1604,-0.029559,0.029559,0.236745,0.08,False,negative
8,moneyline_squared_margin_variance,squared_spread_residual,def_stuff_rate_allowed_delta,Full,1604,0.029559,0.029559,0.236745,0.08,False,positive
9,moneyline_squared_margin_variance,squared_spread_residual,rush_rate_std_downs_delta,Full,1604,-0.027918,0.027918,0.263798,0.08,False,negative



Moneyline threshold clear counts by variance target and stratum:


,outcome,stratum,clears_threshold
0,moneyline_abs_margin_variance,ACC,5
1,moneyline_abs_margin_variance,American Athletic,5
2,moneyline_abs_margin_variance,Big 12,7
3,moneyline_abs_margin_variance,Big Ten,13
4,moneyline_abs_margin_variance,Conference USA,8
5,moneyline_abs_margin_variance,Full,2
6,moneyline_abs_margin_variance,G5,0
7,moneyline_abs_margin_variance,Mid-American,9
8,moneyline_abs_margin_variance,Mountain West,11
9,moneyline_abs_margin_variance,P4,2



All signal rows after adding moneyline: 1248
All outcomes:
outcome
spread                               312
over_under                           312
moneyline_abs_margin_variance        312
moneyline_squared_margin_variance    312

Cell 5B complete.
Use all_signal, not primary_signal alone, for later verdict construction.
Share the full-population moneyline tables and threshold counts before Cell 6.


In [36]:
# Cell 6 — Within-season trajectory signal tests by conference game bucket

# -------------------------------------------------------------------
# Compute conference game number for each team, then assign home-team bucket
# -------------------------------------------------------------------

team_game_order = pd.concat(
    [
        valid_games[
            [
                "game_id",
                "season",
                "week",
                "start_date",
                "home_team",
                "home_conference",
            ]
        ].rename(
            columns={
                "home_team": "team_name",
                "home_conference": "conference",
            }
        ),
        valid_games[
            [
                "game_id",
                "season",
                "week",
                "start_date",
                "away_team",
                "away_conference",
            ]
        ].rename(
            columns={
                "away_team": "team_name",
                "away_conference": "conference",
            }
        ),
    ],
    ignore_index=True,
)

team_game_order = team_game_order.sort_values(
    ["team_name", "season", "week", "start_date", "game_id"]
).copy()

team_game_order["team_conf_game_number"] = (
    team_game_order
    .groupby(["team_name", "season"])
    .cumcount()
    + 1
)

home_game_number = team_game_order.merge(
    valid_games[["game_id", "home_team"]],
    on="game_id",
    how="inner",
)

home_game_number = home_game_number[
    home_game_number["team_name"] == home_game_number["home_team"]
][
    ["game_id", "team_conf_game_number"]
].copy()

style_delta_analysis = style_delta_analysis.merge(
    home_game_number,
    on="game_id",
    how="left",
)

def assign_conf_game_bucket(n):
    if pd.isna(n):
        return np.nan
    if n == 1:
        return "conf_game_1"
    if 2 <= n <= 4:
        return "conf_games_2_4"
    if 5 <= n <= 8:
        return "conf_games_5_8"
    if 9 <= n <= 12:
        return "conf_games_9_12"
    return "conf_game_13_plus"

style_delta_analysis["conf_game_bucket"] = style_delta_analysis["team_conf_game_number"].apply(assign_conf_game_bucket)

print("Conference game bucket distribution:")
print(style_delta_analysis["conf_game_bucket"].value_counts().sort_index().to_string())

print("\nNull conference game number rows:", style_delta_analysis["team_conf_game_number"].isna().sum())

assert style_delta_analysis["team_conf_game_number"].isna().sum() == 0, "ERROR: Null conference game number found."

# -------------------------------------------------------------------
# Run trajectory tests
# -------------------------------------------------------------------

TRAJECTORY_BUCKETS = [
    "conf_game_1",
    "conf_games_2_4",
    "conf_games_5_8",
    "conf_games_9_12",
]

trajectory_rows = []

for bucket in TRAJECTORY_BUCKETS:
    bucket_df = style_delta_analysis[
        style_delta_analysis["conf_game_bucket"] == bucket
    ].copy()
    
    spread_bucket = run_signal_tests(
        df=bucket_df,
        feature_cols=STYLE_DELTA_COLS,
        outcome_col="point_differential",
        outcome_label="spread",
        control_cols=PRIMARY_CONTROLS,
    )
    spread_bucket["trajectory_bucket"] = bucket
    
    total_bucket = run_signal_tests(
        df=bucket_df,
        feature_cols=STYLE_DELTA_COLS,
        outcome_col="total_points",
        outcome_label="over_under",
        control_cols=PRIMARY_CONTROLS,
    )
    total_bucket["trajectory_bucket"] = bucket
    
    abs_bucket = run_signal_tests(
        df=bucket_df,
        feature_cols=STYLE_DELTA_COLS,
        outcome_col="abs_spread_residual",
        outcome_label="moneyline_abs_margin_variance",
        control_cols=PRIMARY_CONTROLS,
    )
    abs_bucket["trajectory_bucket"] = bucket
    
    sq_bucket = run_signal_tests(
        df=bucket_df,
        feature_cols=STYLE_DELTA_COLS,
        outcome_col="squared_spread_residual",
        outcome_label="moneyline_squared_margin_variance",
        control_cols=PRIMARY_CONTROLS,
    )
    sq_bucket["trajectory_bucket"] = bucket
    
    trajectory_rows.extend([spread_bucket, total_bucket, abs_bucket, sq_bucket])

trajectory_signal = pd.concat(trajectory_rows, ignore_index=True)

print("\nTrajectory signal rows:", len(trajectory_signal))
print("\nRows by outcome:")
print(trajectory_signal["outcome"].value_counts().to_string())

print("\nRows by bucket:")
print(trajectory_signal["trajectory_bucket"].value_counts().sort_index().to_string())

print("\nFull population threshold clears by outcome and bucket:")
display(
    trajectory_signal[
        trajectory_signal["stratum"] == "Full"
    ]
    .groupby(["outcome", "trajectory_bucket"], as_index=False)["clears_threshold"]
    .sum()
    .sort_values(["outcome", "trajectory_bucket"])
)

print("\nSpread trajectory — Full population, threshold-clearing rows:")
display(
    trajectory_signal[
        (trajectory_signal["stratum"] == "Full")
        & (trajectory_signal["outcome"] == "spread")
        & (trajectory_signal["clears_threshold"])
    ]
    .sort_values(["trajectory_bucket", "abs_partial_r"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nOver/under trajectory — Full population, threshold-clearing rows:")
display(
    trajectory_signal[
        (trajectory_signal["stratum"] == "Full")
        & (trajectory_signal["outcome"] == "over_under")
        & (trajectory_signal["clears_threshold"])
    ]
    .sort_values(["trajectory_bucket", "abs_partial_r"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nMoneyline trajectory — Full population, threshold-clearing rows:")
display(
    trajectory_signal[
        (trajectory_signal["stratum"] == "Full")
        & (trajectory_signal["outcome"].isin([
            "moneyline_abs_margin_variance",
            "moneyline_squared_margin_variance",
        ]))
        & (trajectory_signal["clears_threshold"])
    ]
    .sort_values(["outcome", "trajectory_bucket", "abs_partial_r"], ascending=[True, True, False])
    .reset_index(drop=True)
)

assert set(trajectory_signal["trajectory_bucket"]) == set(TRAJECTORY_BUCKETS), "ERROR: Missing trajectory bucket."
assert set(trajectory_signal["outcome"]) == {
    "spread",
    "over_under",
    "moneyline_abs_margin_variance",
    "moneyline_squared_margin_variance",
}, "ERROR: Missing trajectory outcome."

print("\nCell 6 complete.")
print("Share bucket distribution and full-population threshold-clearing tables before Cell 7.")

Conference game bucket distribution:
conf_game_bucket
conf_game_1        186
conf_games_2_4     579
conf_games_5_8     755
conf_games_9_12     84

Null conference game number rows: 0

Trajectory signal rows: 4992

Rows by outcome:
outcome
spread                               1248
over_under                           1248
moneyline_abs_margin_variance        1248
moneyline_squared_margin_variance    1248

Rows by bucket:
trajectory_bucket
conf_game_1        1248
conf_games_2_4     1248
conf_games_5_8     1248
conf_games_9_12    1248

Full population threshold clears by outcome and bucket:


,outcome,trajectory_bucket,clears_threshold
0,moneyline_abs_margin_variance,conf_game_1,3
1,moneyline_abs_margin_variance,conf_games_2_4,3
2,moneyline_abs_margin_variance,conf_games_5_8,3
3,moneyline_abs_margin_variance,conf_games_9_12,15
4,moneyline_squared_margin_variance,conf_game_1,10
5,moneyline_squared_margin_variance,conf_games_2_4,2
6,moneyline_squared_margin_variance,conf_games_5_8,2
7,moneyline_squared_margin_variance,conf_games_9_12,13
8,over_under,conf_game_1,10
9,over_under,conf_games_2_4,0



Spread trajectory — Full population, threshold-clearing rows:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction,trajectory_bucket
0,spread,point_differential,rush_rate_std_downs_delta,Full,186,0.296514,0.296514,3.966005e-05,0.08,True,positive,conf_game_1
1,spread,point_differential,off_pts_per_opportunity_delta,Full,185,0.283500,0.283500,9.213487e-05,0.08,True,positive,conf_game_1
2,spread,point_differential,def_pts_per_opportunity_allowed_delta,Full,185,-0.283500,0.283500,9.213487e-05,0.08,True,negative,conf_game_1
3,spread,point_differential,rush_rate_pass_downs_delta,Full,186,0.215201,0.215201,3.179737e-03,0.08,True,positive,conf_game_1
4,spread,point_differential,off_epa_pass_delta,Full,186,0.162448,0.162448,2.673970e-02,0.08,True,positive,conf_game_1
5,spread,point_differential,def_epa_pass_allowed_delta,Full,186,-0.162448,0.162448,2.673970e-02,0.08,True,negative,conf_game_1
6,spread,point_differential,off_line_yards_per_rush_delta,Full,186,-0.138874,0.138874,5.870532e-02,0.08,True,negative,conf_game_1
7,spread,point_differential,off_success_rate_pass_delta,Full,186,0.136836,0.136836,6.254984e-02,0.08,True,positive,conf_game_1
8,spread,point_differential,def_success_rate_pass_allowed_delta,Full,186,-0.136836,0.136836,6.254984e-02,0.08,True,negative,conf_game_1
9,spread,point_differential,off_success_rate_pass_downs_delta,Full,186,0.089211,0.089211,2.259365e-01,0.08,True,positive,conf_game_1



Over/under trajectory — Full population, threshold-clearing rows:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction,trajectory_bucket
0,over_under,total_points,off_sack_rate_allowed_delta,Full,186,0.146363,0.146363,0.046215,0.08,True,positive,conf_game_1
1,over_under,total_points,def_sack_rate_delta,Full,186,-0.146363,0.146363,0.046215,0.08,True,negative,conf_game_1
2,over_under,total_points,off_success_rate_pass_downs_delta,Full,186,-0.118590,0.118590,0.106930,0.08,True,negative,conf_game_1
3,over_under,total_points,off_pts_per_opportunity_delta,Full,185,-0.096627,0.096627,0.190729,0.08,True,negative,conf_game_1
4,over_under,total_points,def_pts_per_opportunity_allowed_delta,Full,185,0.096627,0.096627,0.190729,0.08,True,positive,conf_game_1
5,over_under,total_points,off_epa_rush_delta,Full,186,-0.096502,0.096502,0.190094,0.08,True,negative,conf_game_1
6,over_under,total_points,def_epa_rush_allowed_delta,Full,186,0.096502,0.096502,0.190094,0.08,True,positive,conf_game_1
7,over_under,total_points,off_success_rate_pass_delta,Full,186,-0.092598,0.092598,0.208733,0.08,True,negative,conf_game_1
8,over_under,total_points,def_success_rate_pass_allowed_delta,Full,186,0.092598,0.092598,0.208733,0.08,True,positive,conf_game_1
9,over_under,total_points,rush_rate_pass_downs_delta,Full,186,0.083995,0.083995,0.254357,0.08,True,positive,conf_game_1



Moneyline trajectory — Full population, threshold-clearing rows:


,outcome,outcome_col,feature_delta,stratum,n,partial_r,abs_partial_r,p_value,threshold,clears_threshold,direction,trajectory_bucket
0,moneyline_abs_margin_variance,abs_spread_residual,off_sack_rate_allowed_delta,Full,186,0.132265,0.132265,0.071923,0.08,True,positive,conf_game_1
1,moneyline_abs_margin_variance,abs_spread_residual,def_sack_rate_delta,Full,186,-0.132265,0.132265,0.071923,0.08,True,negative,conf_game_1
2,moneyline_abs_margin_variance,abs_spread_residual,off_success_rate_pass_downs_delta,Full,186,0.123031,0.123031,0.094334,0.08,True,positive,conf_game_1
3,moneyline_abs_margin_variance,abs_spread_residual,off_sack_rate_allowed_delta,Full,579,0.120965,0.120965,0.003555,0.08,True,positive,conf_games_2_4
4,moneyline_abs_margin_variance,abs_spread_residual,def_sack_rate_delta,Full,579,-0.120965,0.120965,0.003555,0.08,True,negative,conf_games_2_4
5,moneyline_abs_margin_variance,abs_spread_residual,rush_rate_std_downs_delta,Full,579,-0.086227,0.086227,0.038059,0.08,True,negative,conf_games_2_4
6,moneyline_abs_margin_variance,abs_spread_residual,off_success_rate_pass_downs_delta,Full,755,0.083664,0.083664,0.021501,0.08,True,positive,conf_games_5_8
7,moneyline_abs_margin_variance,abs_spread_residual,off_explosive_rate_20_delta,Full,755,-0.081604,0.081604,0.024943,0.08,True,negative,conf_games_5_8
8,moneyline_abs_margin_variance,abs_spread_residual,def_explosive_rate_20_allowed_delta,Full,755,0.081604,0.081604,0.024943,0.08,True,positive,conf_games_5_8
9,moneyline_abs_margin_variance,abs_spread_residual,off_explosive_rate_10_delta,Full,84,0.218126,0.218126,0.046231,0.08,True,positive,conf_games_9_12



Cell 6 complete.
Share bucket distribution and full-population threshold-clearing tables before Cell 7.


In [37]:
# Cell 7 — YoY stability for style dimensions at team-season level
# YoY stability is for prior construction only.
# It does not gate game-level style signal testing.

team_season_style = (
    team_game_style
    .groupby(["team_name", "season"], as_index=False)[style_metric_cols]
    .mean()
)

print("Team-season style rows:", len(team_season_style))
print("\nRows by season:")
print(team_season_style["season"].value_counts().sort_index().to_string())

print("\nUnique teams by season:")
print(team_season_style.groupby("season")["team_name"].nunique().to_string())

assert HOLDOUT_SEASON not in team_season_style["season"].unique(), "ERROR: 2025 holdout leaked into team_season_style."

def yoy_corr_for_feature(df, feature_col, season_a, season_b):
    """
    Compute YoY Pearson correlation for a feature between two adjacent seasons.
    """
    a = df[df["season"] == season_a][["team_name", feature_col]].rename(
        columns={feature_col: f"{feature_col}_{season_a}"}
    )
    b = df[df["season"] == season_b][["team_name", feature_col]].rename(
        columns={feature_col: f"{feature_col}_{season_b}"}
    )
    
    joined = a.merge(b, on="team_name", how="inner")
    joined = joined.replace([np.inf, -np.inf], np.nan).dropna()
    
    n = len(joined)
    
    if n < 10:
        return {
            "n": n,
            "r": np.nan,
            "p_value": np.nan,
        }
    
    r, p = stats.pearsonr(
        joined[f"{feature_col}_{season_a}"],
        joined[f"{feature_col}_{season_b}"],
    )
    
    return {
        "n": n,
        "r": r,
        "p_value": p,
    }

yoy_rows = []

for feature_col in style_metric_cols:
    r_22_23 = yoy_corr_for_feature(team_season_style, feature_col, 2022, 2023)
    r_23_24 = yoy_corr_for_feature(team_season_style, feature_col, 2023, 2024)
    
    valid_rs = [
        r_22_23["r"],
        r_23_24["r"],
    ]
    valid_rs = [r for r in valid_rs if pd.notna(r)]
    
    avg_yoy_r = np.mean(valid_rs) if len(valid_rs) > 0 else np.nan
    
    yoy_rows.append({
        "feature": feature_col,
        "n_2022_2023": r_22_23["n"],
        "r_2022_2023": r_22_23["r"],
        "p_2022_2023": r_22_23["p_value"],
        "n_2023_2024": r_23_24["n"],
        "r_2023_2024": r_23_24["r"],
        "p_2023_2024": r_23_24["p_value"],
        "avg_yoy_r": avg_yoy_r,
        "stable_for_prior_seed": (
            avg_yoy_r >= 0.40
            if pd.notna(avg_yoy_r)
            else False
        ),
        "strongly_stable_for_prior_seed": (
            avg_yoy_r >= 0.60
            if pd.notna(avg_yoy_r)
            else False
        ),
    })

style_yoy = pd.DataFrame(yoy_rows)

print("\nYoY stability sorted by avg_yoy_r:")
display(
    style_yoy
    .sort_values("avg_yoy_r", ascending=False)
    .reset_index(drop=True)
)

print("\nStable-for-prior count:")
print(style_yoy["stable_for_prior_seed"].value_counts().to_string())

print("\nStrongly-stable-for-prior count:")
print(style_yoy["strongly_stable_for_prior_seed"].value_counts().to_string())

assert len(style_yoy) == len(style_metric_cols), "ERROR: YoY result count does not match style metric count."

print("\nCell 7 complete.")
print("Share the YoY stability table before Cell 8.")

Team-season style rows: 384

Rows by season:
season
2022    124
2023    129
2024    131

Unique teams by season:
season
2022    124
2023    129
2024    131

YoY stability sorted by avg_yoy_r:


,feature,n_2022_2023,r_2022_2023,p_2022_2023,n_2023_2024,r_2023_2024,p_2023_2024,avg_yoy_r,stable_for_prior_seed,strongly_stable_for_prior_seed
0,rush_rate_std_downs,124,0.496828,4.382789e-09,129,0.481267,7.770029e-09,0.489047,True,False
1,rush_rate_pass_downs,124,0.544802,6.097380e-11,129,0.384760,6.729841e-06,0.464781,True,False
2,def_success_rate_std_downs_allowed,124,0.372683,2.020612e-05,129,0.370697,1.529312e-05,0.371690,False,False
3,def_success_rate_rush_allowed,124,0.420851,1.133513e-06,129,0.267342,2.192617e-03,0.344097,False,False
4,off_success_rate_rush,124,0.367937,2.620802e-05,129,0.290819,8.273263e-04,0.329378,False,False
5,off_explosive_rate_10,124,0.361674,3.671202e-05,129,0.293627,7.320985e-04,0.327651,False,False
6,def_explosive_rate_20_allowed,124,0.330356,1.788184e-04,129,0.318787,2.313863e-04,0.324572,False,False
7,off_success_rate_std_downs,124,0.413238,1.842368e-06,129,0.223360,1.094579e-02,0.318299,False,False
8,def_explosive_rate_10_allowed,124,0.401659,3.770958e-06,129,0.218055,1.304728e-02,0.309857,False,False
9,def_epa_rush_allowed,124,0.350530,6.574038e-05,129,0.267586,2.171519e-03,0.309058,False,False



Stable-for-prior count:
stable_for_prior_seed
False    22
True      2

Strongly-stable-for-prior count:
strongly_stable_for_prior_seed
False    24

Cell 7 complete.
Share the YoY stability table before Cell 8.


In [38]:
# Cell 6B — Game-level diagnostic examples for strongest style mismatch signals
# This does not replace partial-r testing.
# It verifies that the strongest statistical signals map to actual games sensibly.

GAME_DIAGNOSTIC_FEATURES = [
    "rush_rate_std_downs_delta",
    "rush_rate_pass_downs_delta",
    "off_success_rate_pass_delta",
    "def_success_rate_pass_allowed_delta",
    "off_pts_per_opportunity_delta",
    "def_pts_per_opportunity_allowed_delta",
    "off_sack_rate_allowed_delta",
    "def_sack_rate_delta",
    "off_epa_pass_delta",
    "def_epa_pass_allowed_delta",
]

diagnostic_cols = [
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "conference",
    "tier",
    "home_points",
    "away_points",
    "point_differential",
    "total_points",
    "sp_rating_delta",
    "close_game_epa_delta",
    "conf_game_bucket",
] + GAME_DIAGNOSTIC_FEATURES

game_diagnostics = style_delta_analysis[diagnostic_cols].copy()

print("Game diagnostic rows:", len(game_diagnostics))
print("Unique games:", game_diagnostics["game_id"].nunique())

assert len(game_diagnostics) == len(style_delta_analysis), "ERROR: Diagnostic row count mismatch."

for feature in GAME_DIAGNOSTIC_FEATURES:
    print(f"\n\n==============================")
    print(f"Top positive games for {feature}")
    print(f"==============================")
    display(
        game_diagnostics
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[feature])
        .sort_values(feature, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    
    print(f"\n==============================")
    print(f"Top negative games for {feature}")
    print(f"==============================")
    display(
        game_diagnostics
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[feature])
        .sort_values(feature, ascending=True)
        .head(10)
        .reset_index(drop=True)
    )

print("\nCell 6B complete.")
print("Review whether the largest style mismatches look like real football signal or data artifacts.")

Game diagnostic rows: 1604
Unique games: 1604


Top positive games for rush_rate_std_downs_delta


,game_id,season,week,home_team,away_team,conference,tier,home_points,away_points,point_differential,...,rush_rate_std_downs_delta,rush_rate_pass_downs_delta,off_success_rate_pass_delta,def_success_rate_pass_allowed_delta,off_pts_per_opportunity_delta,def_pts_per_opportunity_allowed_delta,off_sack_rate_allowed_delta,def_sack_rate_delta,off_epa_pass_delta,def_epa_pass_allowed_delta
0,401415646,2022,9,Navy,Temple,American Athletic,G5,27,20,7,...,0.692308,0.772727,-0.318182,0.318182,1.250000,-1.250000,0.310606,-0.310606,-1.143436,1.143436
1,401645328,2024,4,Army,Rice,American Athletic,G5,37,14,23,...,0.642857,0.223529,-0.011364,0.011364,0.000000,0.000000,0.000000,0.000000,-0.056534,0.056534
2,401415262,2022,12,Air Force,Colorado State,Mountain West,G5,24,12,12,...,0.641705,0.595238,-0.419355,0.419355,0.500000,-0.500000,0.107527,-0.107527,-0.863503,0.863503
3,401643751,2024,11,Air Force,Fresno State,Mountain West,G5,36,28,8,...,0.625238,0.707071,-0.222222,0.222222,-1.392857,1.392857,-0.074074,0.074074,-0.548328,0.548328
4,401531417,2023,12,Navy,East Carolina,American Athletic,G5,10,0,10,...,0.615497,0.285714,0.043152,-0.043152,0.000000,0.000000,0.080675,-0.080675,0.337019,-0.337019
5,401415632,2022,6,Navy,Tulsa,American Athletic,G5,53,21,32,...,0.598870,0.442308,-0.266129,0.266129,-2.428571,2.428571,-0.032258,0.032258,-0.684876,0.684876
6,401645324,2024,2,Navy,Temple,American Athletic,G5,38,11,27,...,0.587317,0.476190,0.207729,-0.207729,1.000000,-1.000000,-0.021739,0.021739,0.471475,-0.471475
7,401532581,2023,3,Air Force,Utah State,Mountain West,G5,39,21,18,...,0.578526,0.455882,0.167568,-0.167568,-0.107143,0.107143,0.118919,-0.118919,-0.008157,0.008157
8,401416618,2022,6,Miami (OH),Kent State,Mid-American,G5,27,24,3,...,0.563285,0.021739,-0.035714,0.035714,-1.400000,1.400000,-0.047619,0.047619,0.694820,-0.694820
9,401628489,2024,4,Michigan,USC,Big Ten,P4,27,24,3,...,0.555556,0.268519,-0.148052,0.148052,3.500000,-3.500000,0.070130,-0.070130,-0.492498,0.492498



Top negative games for rush_rate_std_downs_delta


,game_id,season,week,home_team,away_team,conference,tier,home_points,away_points,point_differential,...,rush_rate_std_downs_delta,rush_rate_pass_downs_delta,off_success_rate_pass_delta,def_success_rate_pass_allowed_delta,off_pts_per_opportunity_delta,def_pts_per_opportunity_allowed_delta,off_sack_rate_allowed_delta,def_sack_rate_delta,off_epa_pass_delta,def_epa_pass_allowed_delta
0,401415267,2022,13,San Diego State,Air Force,Mountain West,G5,3,13,-10,...,-0.657011,-0.675439,0.104839,-0.104839,-1.166667,1.166667,-0.185484,0.185484,0.302008,-0.302008
1,401645331,2024,5,Temple,Army,American Athletic,G5,14,42,-28,...,-0.656593,-0.375000,-0.077220,0.077220,-5.833333,5.833333,0.189189,-0.189189,-0.176782,0.176782
2,401415661,2022,12,UCF,Navy,American Athletic,G5,14,17,-3,...,-0.646259,-0.513725,0.324324,-0.324324,-1.500000,1.500000,-0.531532,0.531532,0.332619,-0.332619
3,401532603,2023,9,Colorado State,Air Force,Mountain West,G5,13,30,-17,...,-0.612245,-0.558974,0.196809,-0.196809,-5.400000,5.400000,-0.082447,0.082447,-0.036192,0.036192
4,401532584,2023,4,San José State,Air Force,Mountain West,G5,20,45,-25,...,-0.600233,-0.775000,-0.454545,0.454545,-2.500000,2.500000,0.000000,0.000000,-1.079028,1.079028
5,401629045,2024,11,North Texas,Army,American Athletic,G5,3,14,-11,...,-0.599664,-0.523810,-0.205263,0.205263,-3.500000,3.500000,-0.173684,0.173684,0.468184,-0.468184
6,401415650,2022,10,Cincinnati,Navy,American Athletic,G5,20,10,10,...,-0.588315,-0.446154,0.110714,-0.110714,1.750000,-1.750000,-0.192857,0.192857,-0.036956,0.036956
7,401403935,2022,10,Mississippi State,Auburn,SEC,P4,39,33,6,...,-0.582143,-0.285714,0.253846,-0.253846,-0.142857,0.142857,-0.076923,0.076923,0.580828,-0.580828
8,401645323,2024,2,Florida Atlantic,Army,American Athletic,G5,7,24,-17,...,-0.566278,-0.857143,0.250000,-0.250000,-2.333333,2.333333,0.000000,0.000000,-0.000547,0.000547
9,401643763,2024,13,Nevada,Air Force,Mountain West,G5,19,22,-3,...,-0.536585,-0.567251,0.459459,-0.459459,-1.800000,1.800000,0.054054,-0.054054,1.016193,-1.016193




Top positive games for rush_rate_pass_downs_delta


...



Top positive games for def_epa_pass_allowed_delta


,game_id,season,week,home_team,away_team,conference,tier,home_points,away_points,point_differential,...,rush_rate_std_downs_delta,rush_rate_pass_downs_delta,off_success_rate_pass_delta,def_success_rate_pass_allowed_delta,off_pts_per_opportunity_delta,def_pts_per_opportunity_allowed_delta,off_sack_rate_allowed_delta,def_sack_rate_delta,off_epa_pass_delta,def_epa_pass_allowed_delta
0,401643768,2024,14,San Diego State,Air Force,Mountain West,G5,20,31,-11,...,-0.394928,-0.708333,-0.542857,0.542857,1.200000,-1.200000,0.085714,-0.085714,-5.404397,5.404397
1,401645339,2024,6,Tulsa,Army,American Athletic,G5,7,49,-42,...,-0.300055,-0.075000,-0.472906,0.472906,-4.666667,4.666667,-0.142857,0.142857,-2.115006,2.115006
2,401645332,2024,5,UAB,Navy,American Athletic,G5,18,41,-23,...,-0.217209,0.227273,-0.227273,0.227273,0.200000,-0.200000,0.090909,-0.090909,-1.446404,1.446404
3,401644685,2024,7,Eastern Michigan,Miami (OH),Mid-American,G5,14,38,-24,...,-0.220197,-0.506579,-0.248062,0.248062,-2.450000,2.450000,0.069767,-0.069767,-1.435413,1.435413
4,401645370,2024,12,Navy,Tulane,American Athletic,G5,0,35,-35,...,0.025510,-0.085973,-0.533333,0.533333,-5.833333,5.833333,0.087179,-0.087179,-1.415774,1.415774
5,401628340,2024,2,Kentucky,South Carolina,SEC,P4,6,31,-25,...,0.028788,-0.131944,-0.320588,0.320588,-2.800000,2.800000,0.141176,-0.141176,-1.372835,1.372835
6,401525904,2023,13,Kansas State,Iowa State,Big 12,P4,35,42,-7,...,-0.071093,-0.207143,-0.145833,0.145833,4.142857,-4.142857,0.000000,0.000000,-1.333774,1.333774
7,401539482,2023,14,UNLV,Boise State,Mountain West,G5,20,44,-24,...,-0.368984,-0.261905,-0.316667,0.316667,-2.100000,2.100000,-0.016667,0.016667,-1.273816,1.273816
8,401525903,2023,13,Cincinnati,Kansas,Big 12,P4,16,49,-33,...,-0.039286,-0.194444,-0.409867,0.409867,-1.416667,1.416667,0.000000,0.000000,-1.263883,1.263883
9,401628418,2024,11,Vanderbilt,South Carolina,SEC,P4,7,28,-21,...,-0.231167,-0.110390,-0.316667,0.316667,-2.450000,2.450000,0.060606,-0.060606,-1.182783,1.182783



Top negative games for def_epa_pass_allowed_delta


,game_id,season,week,home_team,away_team,conference,tier,home_points,away_points,point_differential,...,rush_rate_std_downs_delta,rush_rate_pass_downs_delta,off_success_rate_pass_delta,def_success_rate_pass_allowed_delta,off_pts_per_opportunity_delta,def_pts_per_opportunity_allowed_delta,off_sack_rate_allowed_delta,def_sack_rate_delta,off_epa_pass_delta,def_epa_pass_allowed_delta
0,401532589,2023,5,Air Force,San Diego State,Mountain West,G5,49,10,39,...,0.241758,0.690476,0.417143,-0.417143,3.266667,-3.266667,-0.040000,0.040000,2.095715,-2.095715
1,401415237,2022,7,UNLV,Air Force,Mountain West,G5,7,42,-35,...,-0.364146,-0.666667,0.666667,-0.666667,-3.000000,3.000000,0.111111,-0.111111,1.741595,-1.741595
2,401640985,2024,10,Florida International,New Mexico State,Conference USA,G5,34,13,21,...,-0.109768,0.042017,0.309524,-0.309524,0.000000,0.000000,-0.095238,0.095238,1.633362,-1.633362
3,401628491,2024,4,Washington,Northwestern,Big Ten,P4,24,5,19,...,0.177083,-0.473684,0.472747,-0.472747,2.333333,-2.333333,-0.004449,0.004449,1.509109,-1.509109
4,401643835,2024,12,Texas State,Southern Miss,Sun Belt,G5,58,3,55,...,-0.164980,-0.381963,0.474790,-0.474790,4.666667,-4.666667,-0.113445,0.113445,1.478605,-1.478605
5,401673462,2024,15,Army,Tulane,American Athletic,G5,35,14,21,...,0.392208,0.625000,0.480000,-0.480000,2.916667,-2.916667,-0.040000,0.040000,1.376792,-1.376792
6,401525543,2023,10,Louisville,Virginia Tech,ACC,P4,34,3,31,...,0.279561,0.147059,0.412088,-0.412088,2.166667,-2.166667,-0.082418,0.082418,1.353059,-1.353059
7,401532406,2023,4,Toledo,Western Michigan,Mid-American,G5,49,31,18,...,-0.016596,0.109524,0.369048,-0.369048,1.888889,-1.888889,-0.125000,0.125000,1.272042,-1.272042
8,401532440,2023,11,Toledo,Eastern Michigan,Mid-American,G5,49,23,26,...,-0.039683,-0.106952,0.547009,-0.547009,1.750000,-1.750000,-0.002849,0.002849,1.256662,-1.256662
9,401645364,2024,11,Tulane,Temple,American Athletic,G5,52,6,46,...,0.254944,0.020243,0.430070,-0.430070,4.666667,-4.666667,-0.153846,0.153846,1.251738,-1.251738



Cell 6B complete.
Review whether the largest style mismatches look like real football signal or data artifacts.


In [39]:
# Cell 8 — Consolidate signal, trajectory, and YoY into feature-level summary

# -------------------------------------------------------------------
# Helper: summarize threshold-clearing strata for each feature/outcome
# -------------------------------------------------------------------

def summarize_signal_by_feature(signal_df, outcome_name):
    """
    Summarize primary all-strata signal results for a single outcome.
    Uses all_signal, which includes spread, over/under, and moneyline variance targets.
    """
    d = signal_df[signal_df["outcome"] == outcome_name].copy()
    
    rows = []
    
    for feature in sorted(d["feature_delta"].unique()):
        fd = d[d["feature_delta"] == feature].copy()
        
        full_row = fd[fd["stratum"] == "Full"].copy()
        p4_row = fd[fd["stratum"] == "P4"].copy()
        g5_row = fd[fd["stratum"] == "G5"].copy()
        
        full_r = full_row["partial_r"].iloc[0] if len(full_row) == 1 else np.nan
        full_n = full_row["n"].iloc[0] if len(full_row) == 1 else np.nan
        full_clears = full_row["clears_threshold"].iloc[0] if len(full_row) == 1 else False
        
        p4_r = p4_row["partial_r"].iloc[0] if len(p4_row) == 1 else np.nan
        p4_clears = p4_row["clears_threshold"].iloc[0] if len(p4_row) == 1 else False
        
        g5_r = g5_row["partial_r"].iloc[0] if len(g5_row) == 1 else np.nan
        g5_clears = g5_row["clears_threshold"].iloc[0] if len(g5_row) == 1 else False
        
        conf_rows = fd[
            ~fd["stratum"].isin(["Full", "P4", "G5"])
        ].copy()
        
        clearing_confs = (
            conf_rows[conf_rows["clears_threshold"]]
            .sort_values("abs_partial_r", ascending=False)["stratum"]
            .tolist()
        )
        
        rows.append({
            "feature_delta": feature,
            "outcome": outcome_name,
            "full_n": full_n,
            "full_partial_r": full_r,
            "full_abs_partial_r": abs(full_r) if pd.notna(full_r) else np.nan,
            "full_clears_threshold": bool(full_clears),
            "p4_partial_r": p4_r,
            "p4_clears_threshold": bool(p4_clears),
            "g5_partial_r": g5_r,
            "g5_clears_threshold": bool(g5_clears),
            "num_conferences_clearing": len(clearing_confs),
            "conferences_clearing": ", ".join(clearing_confs),
        })
    
    return pd.DataFrame(rows)


spread_summary = summarize_signal_by_feature(all_signal, "spread")
ou_summary = summarize_signal_by_feature(all_signal, "over_under")
ml_abs_summary = summarize_signal_by_feature(all_signal, "moneyline_abs_margin_variance")
ml_sq_summary = summarize_signal_by_feature(all_signal, "moneyline_squared_margin_variance")

signal_summary_long = pd.concat(
    [
        spread_summary,
        ou_summary,
        ml_abs_summary,
        ml_sq_summary,
    ],
    ignore_index=True,
)

print("Signal summary rows:", len(signal_summary_long))
print("\nRows by outcome:")
print(signal_summary_long["outcome"].value_counts().to_string())

print("\nFull-population clearing counts by outcome:")
print(
    signal_summary_long
    .groupby("outcome")["full_clears_threshold"]
    .sum()
    .sort_index()
    .to_string()
)

# -------------------------------------------------------------------
# Trajectory summary
# -------------------------------------------------------------------

trajectory_full = trajectory_signal[
    trajectory_signal["stratum"] == "Full"
].copy()

trajectory_feature_summary = (
    trajectory_full
    .groupby(["feature_delta", "outcome"], as_index=False)
    .agg(
        trajectory_buckets_clearing=("clears_threshold", "sum"),
        max_trajectory_abs_partial_r=("abs_partial_r", "max"),
    )
)

trajectory_bucket_lists = (
    trajectory_full[trajectory_full["clears_threshold"]]
    .groupby(["feature_delta", "outcome"])["trajectory_bucket"]
    .apply(lambda x: ", ".join(sorted(x.unique())))
    .reset_index()
    .rename(columns={"trajectory_bucket": "trajectory_buckets_clearing_list"})
)

trajectory_feature_summary = trajectory_feature_summary.merge(
    trajectory_bucket_lists,
    on=["feature_delta", "outcome"],
    how="left",
)

trajectory_feature_summary["trajectory_buckets_clearing_list"] = (
    trajectory_feature_summary["trajectory_buckets_clearing_list"].fillna("")
)

print("\nTrajectory feature summary rows:", len(trajectory_feature_summary))

# -------------------------------------------------------------------
# YoY summary mapped to delta feature names
# -------------------------------------------------------------------

style_yoy_for_merge = style_yoy.copy()
style_yoy_for_merge["feature_delta"] = style_yoy_for_merge["feature"] + "_delta"

style_yoy_for_merge = style_yoy_for_merge[
    [
        "feature_delta",
        "avg_yoy_r",
        "stable_for_prior_seed",
        "strongly_stable_for_prior_seed",
    ]
].copy()

# -------------------------------------------------------------------
# Build wide feature-level summary
# -------------------------------------------------------------------

feature_summary = pd.DataFrame({
    "feature_delta": STYLE_DELTA_COLS
})

# Spread
feature_summary = feature_summary.merge(
    spread_summary[
        [
            "feature_delta",
            "full_partial_r",
            "full_abs_partial_r",
            "full_clears_threshold",
            "p4_partial_r",
            "p4_clears_threshold",
            "g5_partial_r",
            "g5_clears_threshold",
            "num_conferences_clearing",
            "conferences_clearing",
        ]
    ].rename(columns={
        "full_partial_r": "spread_full_partial_r",
        "full_abs_partial_r": "spread_full_abs_partial_r",
        "full_clears_threshold": "spread_full_clears_threshold",
        "p4_partial_r": "spread_p4_partial_r",
        "p4_clears_threshold": "spread_p4_clears_threshold",
        "g5_partial_r": "spread_g5_partial_r",
        "g5_clears_threshold": "spread_g5_clears_threshold",
        "num_conferences_clearing": "spread_num_conferences_clearing",
        "conferences_clearing": "spread_conferences_clearing",
    }),
    on="feature_delta",
    how="left",
)

# Over/under
feature_summary = feature_summary.merge(
    ou_summary[
        [
            "feature_delta",
            "full_partial_r",
            "full_abs_partial_r",
            "full_clears_threshold",
            "p4_partial_r",
            "p4_clears_threshold",
            "g5_partial_r",
            "g5_clears_threshold",
            "num_conferences_clearing",
            "conferences_clearing",
        ]
    ].rename(columns={
        "full_partial_r": "ou_full_partial_r",
        "full_abs_partial_r": "ou_full_abs_partial_r",
        "full_clears_threshold": "ou_full_clears_threshold",
        "p4_partial_r": "ou_p4_partial_r",
        "p4_clears_threshold": "ou_p4_clears_threshold",
        "g5_partial_r": "ou_g5_partial_r",
        "g5_clears_threshold": "ou_g5_clears_threshold",
        "num_conferences_clearing": "ou_num_conferences_clearing",
        "conferences_clearing": "ou_conferences_clearing",
    }),
    on="feature_delta",
    how="left",
)

# Moneyline abs
feature_summary = feature_summary.merge(
    ml_abs_summary[
        [
            "feature_delta",
            "full_partial_r",
            "full_abs_partial_r",
            "full_clears_threshold",
            "num_conferences_clearing",
            "conferences_clearing",
        ]
    ].rename(columns={
        "full_partial_r": "ml_abs_full_partial_r",
        "full_abs_partial_r": "ml_abs_full_abs_partial_r",
        "full_clears_threshold": "ml_abs_full_clears_threshold",
        "num_conferences_clearing": "ml_abs_num_conferences_clearing",
        "conferences_clearing": "ml_abs_conferences_clearing",
    }),
    on="feature_delta",
    how="left",
)

# Moneyline squared
feature_summary = feature_summary.merge(
    ml_sq_summary[
        [
            "feature_delta",
            "full_partial_r",
            "full_abs_partial_r",
            "full_clears_threshold",
            "num_conferences_clearing",
            "conferences_clearing",
        ]
    ].rename(columns={
        "full_partial_r": "ml_sq_full_partial_r",
        "full_abs_partial_r": "ml_sq_full_abs_partial_r",
        "full_clears_threshold": "ml_sq_full_clears_threshold",
        "num_conferences_clearing": "ml_sq_num_conferences_clearing",
        "conferences_clearing": "ml_sq_conferences_clearing",
    }),
    on="feature_delta",
    how="left",
)

# Trajectory summaries by outcome
for outcome_name, prefix in [
    ("spread", "spread"),
    ("over_under", "ou"),
    ("moneyline_abs_margin_variance", "ml_abs"),
    ("moneyline_squared_margin_variance", "ml_sq"),
]:
    temp = trajectory_feature_summary[
        trajectory_feature_summary["outcome"] == outcome_name
    ][
        [
            "feature_delta",
            "trajectory_buckets_clearing",
            "max_trajectory_abs_partial_r",
            "trajectory_buckets_clearing_list",
        ]
    ].rename(columns={
        "trajectory_buckets_clearing": f"{prefix}_trajectory_buckets_clearing",
        "max_trajectory_abs_partial_r": f"{prefix}_max_trajectory_abs_partial_r",
        "trajectory_buckets_clearing_list": f"{prefix}_trajectory_buckets_clearing_list",
    })
    
    feature_summary = feature_summary.merge(
        temp,
        on="feature_delta",
        how="left",
    )

# YoY
feature_summary = feature_summary.merge(
    style_yoy_for_merge,
    on="feature_delta",
    how="left",
)

# Composite flags
feature_summary["any_spread_signal"] = (
    feature_summary["spread_full_clears_threshold"]
    | (feature_summary["spread_num_conferences_clearing"] > 0)
    | (feature_summary["spread_trajectory_buckets_clearing"] > 0)
)

feature_summary["any_ou_signal"] = (
    feature_summary["ou_full_clears_threshold"]
    | (feature_summary["ou_num_conferences_clearing"] > 0)
    | (feature_summary["ou_trajectory_buckets_clearing"] > 0)
)

feature_summary["any_moneyline_signal"] = (
    feature_summary["ml_abs_full_clears_threshold"]
    | feature_summary["ml_sq_full_clears_threshold"]
    | (feature_summary["ml_abs_num_conferences_clearing"] > 0)
    | (feature_summary["ml_sq_num_conferences_clearing"] > 0)
    | (feature_summary["ml_abs_trajectory_buckets_clearing"] > 0)
    | (feature_summary["ml_sq_trajectory_buckets_clearing"] > 0)
)

feature_summary["any_game_level_signal"] = (
    feature_summary["any_spread_signal"]
    | feature_summary["any_ou_signal"]
    | feature_summary["any_moneyline_signal"]
)

print("\nFeature summary rows:", len(feature_summary))

print("\nFeature summary sorted by spread full abs partial r:")
display(
    feature_summary
    .sort_values("spread_full_abs_partial_r", ascending=False)
    .reset_index(drop=True)
)

assert len(feature_summary) == len(STYLE_DELTA_COLS), "ERROR: Feature summary row count mismatch."

print("\nCell 8 complete.")

Signal summary rows: 96

Rows by outcome:
outcome
spread                               24
over_under                           24
moneyline_abs_margin_variance        24
moneyline_squared_margin_variance    24

Full-population clearing counts by outcome:
outcome
moneyline_abs_margin_variance        2
moneyline_squared_margin_variance    0
over_under                           0
spread                               8

Trajectory feature summary rows: 96

Feature summary rows: 24

Feature summary sorted by spread full abs partial r:


,feature_delta,spread_full_partial_r,spread_full_abs_partial_r,spread_full_clears_threshold,spread_p4_partial_r,spread_p4_clears_threshold,spread_g5_partial_r,spread_g5_clears_threshold,spread_num_conferences_clearing,spread_conferences_clearing,...,ml_sq_trajectory_buckets_clearing,ml_sq_max_trajectory_abs_partial_r,ml_sq_trajectory_buckets_clearing_list,avg_yoy_r,stable_for_prior_seed,strongly_stable_for_prior_seed,any_spread_signal,any_ou_signal,any_moneyline_signal,any_game_level_signal
0,rush_rate_std_downs_delta,0.292691,0.292691,True,0.335206,True,0.264549,True,9,"Mid-American, SEC, Big Ten, American Athletic,...",...,1,0.150808,conf_games_9_12,0.489047,True,False,True,True,True,True
1,rush_rate_pass_downs_delta,0.216574,0.216574,True,0.265692,True,0.182664,True,9,"SEC, Pac-12, ACC, Big Ten, Mid-American, Mount...",...,1,0.085657,conf_game_1,0.464781,True,False,True,True,True,True
2,off_pts_per_opportunity_delta,0.146895,0.146895,True,0.177533,True,0.118969,True,7,"SEC, Conference USA, Big Ten, Big 12, Sun Belt...",...,2,0.136534,"conf_game_1, conf_games_9_12",0.170339,False,False,True,True,True,True
3,def_pts_per_opportunity_allowed_delta,-0.146895,0.146895,True,-0.177533,True,-0.118969,True,7,"SEC, Conference USA, Big Ten, Big 12, Sun Belt...",...,2,0.136534,"conf_game_1, conf_games_9_12",0.217245,False,False,True,True,True,True
4,def_success_rate_pass_allowed_delta,-0.142680,0.142680,True,-0.163745,True,-0.127372,True,7,"Pac-12, Big 12, Sun Belt, SEC, Conference USA,...",...,1,0.160276,conf_games_9_12,0.272995,False,False,True,True,True,True
5,off_success_rate_pass_delta,0.142680,0.142680,True,0.163745,True,0.127372,True,7,"Pac-12, Big 12, Sun Belt, SEC, Conference USA,...",...,1,0.160276,conf_games_9_12,0.295336,False,False,True,True,True,True
6,def_epa_pass_allowed_delta,-0.088056,0.088056,True,-0.073747,False,-0.101465,True,6,"Sun Belt, Conference USA, SEC, Pac-12, America...",...,0,0.053861,,0.297431,False,False,True,True,True,True
7,off_epa_pass_delta,0.088056,0.088056,True,0.073747,False,0.101465,True,6,"Sun Belt, Conference USA, SEC, Pac-12, America...",...,0,0.053861,,0.275521,False,False,True,True,True,True
8,off_success_rate_pass_downs_delta,0.075526,0.075526,False,0.064213,False,0.093875,True,5,"Conference USA, Sun Belt, Pac-12, Mountain Wes...",...,1,0.099436,conf_game_1,0.253089,False,False,True,True,True,True
9,off_sack_rate_allowed_delta,-0.071307,0.071307,False,-0.043892,False,-0.095096,True,4,"Pac-12, Mid-American, Sun Belt, Big Ten",...,3,0.164510,"conf_game_1, conf_games_2_4, conf_games_9_12",0.243776,False,False,True,True,True,True



Cell 8 complete.


In [40]:
# Cell 9 — Build final style/tempo verdict table

def verdict_from_row(row):
    """
    Assign final EDA verdict for each style dimension.
    
    Important distinction:
    - In-game signal means the dimension matters when observed in a game.
    - Prior seed eligibility requires YoY stability.
    - Pregame deployability still requires a future rolling/current-season version.
    """
    if row["any_game_level_signal"] and row["stable_for_prior_seed"]:
        return "candidate_game_level_signal_and_weak_prior_seed"
    
    if row["any_game_level_signal"]:
        return "candidate_game_level_signal_only"
    
    if row["stable_for_prior_seed"]:
        return "weak_prior_seed_only_no_game_signal"
    
    return "reject_no_reliable_signal"


def spread_verdict_from_row(row):
    if row["spread_full_clears_threshold"]:
        return "full_population_signal"
    if row["spread_p4_clears_threshold"] or row["spread_g5_clears_threshold"]:
        return "tier_specific_signal"
    if row["spread_num_conferences_clearing"] > 0:
        return "conference_specific_signal"
    if row["spread_trajectory_buckets_clearing"] > 0:
        return "trajectory_specific_signal"
    return "no_signal"


def ou_verdict_from_row(row):
    if row["ou_full_clears_threshold"]:
        return "full_population_signal"
    if row["ou_p4_clears_threshold"] or row["ou_g5_clears_threshold"]:
        return "tier_specific_signal"
    if row["ou_num_conferences_clearing"] > 0:
        return "conference_specific_signal"
    if row["ou_trajectory_buckets_clearing"] > 0:
        return "trajectory_specific_signal"
    return "no_signal"


def moneyline_verdict_from_row(row):
    if row["ml_abs_full_clears_threshold"] or row["ml_sq_full_clears_threshold"]:
        return "full_population_variance_signal"
    if (
        row["ml_abs_num_conferences_clearing"] > 0
        or row["ml_sq_num_conferences_clearing"] > 0
    ):
        return "conference_specific_variance_signal"
    if (
        row["ml_abs_trajectory_buckets_clearing"] > 0
        or row["ml_sq_trajectory_buckets_clearing"] > 0
    ):
        return "trajectory_specific_variance_signal"
    return "no_variance_signal"


def prior_verdict_from_row(row):
    if row["strongly_stable_for_prior_seed"]:
        return "strong_prior_seed_candidate"
    if row["stable_for_prior_seed"]:
        return "weak_prior_seed_candidate"
    return "not_prior_seed"


def deployment_role_from_row(row):
    """
    Avoid overclaiming.
    This notebook used in-game style metrics for game-level signal discovery.
    Pregame deployable versions must be tested separately.
    """
    if row["stable_for_prior_seed"] and row["any_game_level_signal"]:
        return "test_prior_and_pregame_rolling_version"
    
    if row["stable_for_prior_seed"]:
        return "test_prior_only_if_model_needs_style_prior"
    
    if row["any_game_level_signal"]:
        return "test_pregame_rolling_current_season_version"
    
    return "do_not_model"


verdict = feature_summary.copy()

verdict["feature"] = verdict["feature_delta"].str.replace("_delta", "", regex=False)

verdict["spread_verdict"] = verdict.apply(spread_verdict_from_row, axis=1)
verdict["over_under_verdict"] = verdict.apply(ou_verdict_from_row, axis=1)
verdict["moneyline_variance_verdict"] = verdict.apply(moneyline_verdict_from_row, axis=1)
verdict["prior_seed_verdict"] = verdict.apply(prior_verdict_from_row, axis=1)
verdict["final_verdict"] = verdict.apply(verdict_from_row, axis=1)
verdict["recommended_deployment_role"] = verdict.apply(deployment_role_from_row, axis=1)

# Human-readable trajectory verdicts
verdict["spread_trajectory_verdict"] = np.where(
    verdict["spread_trajectory_buckets_clearing"] >= 3,
    "holds_broadly_across_season",
    np.where(
        verdict["spread_trajectory_buckets_clearing"] > 0,
        "bucket_specific",
        "no_trajectory_signal",
    )
)

verdict["ou_trajectory_verdict"] = np.where(
    verdict["ou_trajectory_buckets_clearing"] >= 3,
    "holds_broadly_across_season",
    np.where(
        verdict["ou_trajectory_buckets_clearing"] > 0,
        "bucket_specific",
        "no_trajectory_signal",
    )
)

verdict["moneyline_trajectory_verdict"] = np.where(
    (
        verdict["ml_abs_trajectory_buckets_clearing"]
        + verdict["ml_sq_trajectory_buckets_clearing"]
    ) >= 3,
    "holds_or_recurs_across_multiple_buckets",
    np.where(
        (
            verdict["ml_abs_trajectory_buckets_clearing"]
            + verdict["ml_sq_trajectory_buckets_clearing"]
        ) > 0,
        "bucket_specific",
        "no_trajectory_signal",
    )
)

# Explicit methodological note
verdict["methodology_note"] = (
    "Signal tested using in-game home-minus-away style deltas. "
    "This qualifies the football dimension, not direct pregame deployability. "
    "Pregame rolling/current-season versions must be tested before model inclusion."
)

# Reorder columns
verdict_cols = [
    "feature",
    "feature_delta",
    "final_verdict",
    "recommended_deployment_role",
    
    "spread_verdict",
    "spread_full_partial_r",
    "spread_full_abs_partial_r",
    "spread_full_clears_threshold",
    "spread_p4_partial_r",
    "spread_p4_clears_threshold",
    "spread_g5_partial_r",
    "spread_g5_clears_threshold",
    "spread_num_conferences_clearing",
    "spread_conferences_clearing",
    "spread_trajectory_verdict",
    "spread_trajectory_buckets_clearing",
    "spread_trajectory_buckets_clearing_list",
    
    "over_under_verdict",
    "ou_full_partial_r",
    "ou_full_abs_partial_r",
    "ou_full_clears_threshold",
    "ou_p4_partial_r",
    "ou_p4_clears_threshold",
    "ou_g5_partial_r",
    "ou_g5_clears_threshold",
    "ou_num_conferences_clearing",
    "ou_conferences_clearing",
    "ou_trajectory_verdict",
    "ou_trajectory_buckets_clearing",
    "ou_trajectory_buckets_clearing_list",
    
    "moneyline_variance_verdict",
    "ml_abs_full_partial_r",
    "ml_abs_full_abs_partial_r",
    "ml_abs_full_clears_threshold",
    "ml_sq_full_partial_r",
    "ml_sq_full_abs_partial_r",
    "ml_sq_full_clears_threshold",
    "ml_abs_num_conferences_clearing",
    "ml_abs_conferences_clearing",
    "ml_sq_num_conferences_clearing",
    "ml_sq_conferences_clearing",
    "moneyline_trajectory_verdict",
    "ml_abs_trajectory_buckets_clearing",
    "ml_abs_trajectory_buckets_clearing_list",
    "ml_sq_trajectory_buckets_clearing",
    "ml_sq_trajectory_buckets_clearing_list",
    
    "prior_seed_verdict",
    "avg_yoy_r",
    "stable_for_prior_seed",
    "strongly_stable_for_prior_seed",
    
    "methodology_note",
]

verdict = verdict[verdict_cols].copy()

print("Verdict rows:", len(verdict))

print("\nFinal verdict counts:")
print(verdict["final_verdict"].value_counts().to_string())

print("\nRecommended deployment role counts:")
print(verdict["recommended_deployment_role"].value_counts().to_string())

print("\nSpread verdict counts:")
print(verdict["spread_verdict"].value_counts().to_string())

print("\nOver/under verdict counts:")
print(verdict["over_under_verdict"].value_counts().to_string())

print("\nMoneyline variance verdict counts:")
print(verdict["moneyline_variance_verdict"].value_counts().to_string())

print("\nPrior seed verdict counts:")
print(verdict["prior_seed_verdict"].value_counts().to_string())

print("\nFinal verdict table:")
display(
    verdict
    .sort_values(
        [
            "final_verdict",
            "spread_full_abs_partial_r",
            "ml_abs_full_abs_partial_r",
            "ou_full_abs_partial_r",
        ],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

assert len(verdict) == len(STYLE_DELTA_COLS), "ERROR: Verdict row count mismatch."
assert verdict["feature"].isna().sum() == 0, "ERROR: Null feature names found."
assert verdict["final_verdict"].isna().sum() == 0, "ERROR: Null final verdicts found."

print("\nCell 9 complete.")

Verdict rows: 24

Final verdict counts:
final_verdict
candidate_game_level_signal_only                   22
candidate_game_level_signal_and_weak_prior_seed     2

Recommended deployment role counts:
recommended_deployment_role
test_pregame_rolling_current_season_version    22
test_prior_and_pregame_rolling_version          2

Spread verdict counts:
spread_verdict
conference_specific_signal    11
full_population_signal         8
tier_specific_signal           5

Over/under verdict counts:
over_under_verdict
conference_specific_signal    23
tier_specific_signal           1

Moneyline variance verdict counts:
moneyline_variance_verdict
conference_specific_variance_signal    22
full_population_variance_signal         2

Prior seed verdict counts:
prior_seed_verdict
not_prior_seed               22
weak_prior_seed_candidate     2

Final verdict table:


,feature,feature_delta,final_verdict,recommended_deployment_role,spread_verdict,spread_full_partial_r,spread_full_abs_partial_r,spread_full_clears_threshold,spread_p4_partial_r,spread_p4_clears_threshold,...,moneyline_trajectory_verdict,ml_abs_trajectory_buckets_clearing,ml_abs_trajectory_buckets_clearing_list,ml_sq_trajectory_buckets_clearing,ml_sq_trajectory_buckets_clearing_list,prior_seed_verdict,avg_yoy_r,stable_for_prior_seed,strongly_stable_for_prior_seed,methodology_note
0,rush_rate_std_downs,rush_rate_std_downs_delta,candidate_game_level_signal_and_weak_prior_seed,test_prior_and_pregame_rolling_version,full_population_signal,0.292691,0.292691,True,0.335206,True,...,holds_or_recurs_across_multiple_buckets,2,"conf_games_2_4, conf_games_9_12",1,conf_games_9_12,weak_prior_seed_candidate,0.489047,True,False,Signal tested using in-game home-minus-away st...
1,rush_rate_pass_downs,rush_rate_pass_downs_delta,candidate_game_level_signal_and_weak_prior_seed,test_prior_and_pregame_rolling_version,full_population_signal,0.216574,0.216574,True,0.265692,True,...,bucket_specific,1,conf_games_9_12,1,conf_game_1,weak_prior_seed_candidate,0.464781,True,False,Signal tested using in-game home-minus-away st...
2,off_pts_per_opportunity,off_pts_per_opportunity_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.146895,0.146895,True,0.177533,True,...,bucket_specific,0,,2,"conf_game_1, conf_games_9_12",not_prior_seed,0.170339,False,False,Signal tested using in-game home-minus-away st...
3,def_pts_per_opportunity_allowed,def_pts_per_opportunity_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.146895,0.146895,True,-0.177533,True,...,bucket_specific,0,,2,"conf_game_1, conf_games_9_12",not_prior_seed,0.217245,False,False,Signal tested using in-game home-minus-away st...
4,off_success_rate_pass,off_success_rate_pass_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.142680,0.142680,True,0.163745,True,...,bucket_specific,1,conf_games_9_12,1,conf_games_9_12,not_prior_seed,0.295336,False,False,Signal tested using in-game home-minus-away st...
5,def_success_rate_pass_allowed,def_success_rate_pass_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.142680,0.142680,True,-0.163745,True,...,bucket_specific,1,conf_games_9_12,1,conf_games_9_12,not_prior_seed,0.272995,False,False,Signal tested using in-game home-minus-away st...
6,off_epa_pass,off_epa_pass_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.088056,0.088056,True,0.073747,False,...,no_trajectory_signal,0,,0,,not_prior_seed,0.275521,False,False,Signal tested using in-game home-minus-away st...
7,def_epa_pass_allowed,def_epa_pass_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.088056,0.088056,True,-0.073747,False,...,no_trajectory_signal,0,,0,,not_prior_seed,0.297431,False,False,Signal tested using in-game home-minus-away st...
8,off_success_rate_pass_downs,off_success_rate_pass_downs_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,0.075526,0.075526,False,0.064213,False,...,holds_or_recurs_across_multiple_buckets,2,"conf_game_1, conf_games_5_8",1,conf_game_1,not_prior_seed,0.253089,False,False,Signal tested using in-game home-minus-away st...
9,off_sack_rate_allowed,off_sack_rate_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,-0.071307,0.071307,False,-0.043892,False,...,holds_or_recurs_across_multiple_buckets,3,"conf_game_1, conf_games_2_4, conf_games_9_12",3,"conf_game_1, conf_games_2_4, conf_games_9_12",not_prior_seed,0.243776,False,False,Signal tested using in-game home-minus-away st...



Cell 9 complete.


In [41]:
# Cell 10 — Write style/tempo verdict artifacts

from pathlib import Path

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

verdict_path = artifact_dir / "style_tempo_verdict.csv"
summary_path = artifact_dir / "style_tempo_summary.json"
all_signal_path = artifact_dir / "style_tempo_all_signal.csv"
trajectory_path = artifact_dir / "style_tempo_trajectory_signal.csv"
yoy_path = artifact_dir / "style_tempo_yoy_stability.csv"

verdict.to_csv(verdict_path, index=False)
all_signal.to_csv(all_signal_path, index=False)
trajectory_signal.to_csv(trajectory_path, index=False)
style_yoy.to_csv(yoy_path, index=False)

summary_payload = {
    "notebook": "eda_09_style_tempo_delta.ipynb",
    "status": "rebuilt_clean",
    "seasons": DEV_SEASONS,
    "holdout_season_excluded": HOLDOUT_SEASON,
    "base_valid_games": int(len(valid_games)),
    "analysis_games": int(len(style_delta_analysis)),
    "dropped_games_missing_required_controls": int(len(valid_games) - len(style_delta_analysis)),
    "style_dimensions_tested": int(len(style_metric_cols)),
    "style_delta_features_tested": int(len(STYLE_DELTA_COLS)),
    "partial_r_threshold": SIGNAL_THRESHOLD,
    "primary_controls": PRIMARY_CONTROLS,
    "outcomes_tested": [
        "spread",
        "over_under",
        "moneyline_abs_margin_variance",
        "moneyline_squared_margin_variance",
    ],
    "methodology": {
        "grain": "one row per game after home-minus-away style delta construction",
        "raw_metric_grain": "one row per team per game from raw.plays",
        "style_metric_type": "in-game observed style metrics",
        "deployment_warning": (
            "In-game metrics qualify whether dimensions matter. "
            "Pregame rolling/current-season versions must be tested before model deployment."
        ),
        "conference_stratification": "Full, P4, G5, and individual conferences",
        "trajectory_buckets": TRAJECTORY_BUCKETS,
        "yoy_stability_role": "prior seed eligibility only; does not gate game-level signal",
    },
    "final_verdict_counts": verdict["final_verdict"].value_counts().to_dict(),
    "recommended_deployment_role_counts": verdict["recommended_deployment_role"].value_counts().to_dict(),
    "spread_verdict_counts": verdict["spread_verdict"].value_counts().to_dict(),
    "over_under_verdict_counts": verdict["over_under_verdict"].value_counts().to_dict(),
    "moneyline_variance_verdict_counts": verdict["moneyline_variance_verdict"].value_counts().to_dict(),
    "prior_seed_verdict_counts": verdict["prior_seed_verdict"].value_counts().to_dict(),
    "stable_prior_seed_features": (
        verdict[verdict["stable_for_prior_seed"]]["feature"]
        .sort_values()
        .tolist()
    ),
    "full_population_spread_signals": (
        verdict[verdict["spread_full_clears_threshold"]]["feature"]
        .sort_values()
        .tolist()
    ),
    "full_population_over_under_signals": (
        verdict[verdict["ou_full_clears_threshold"]]["feature"]
        .sort_values()
        .tolist()
    ),
    "full_population_moneyline_abs_variance_signals": (
        verdict[verdict["ml_abs_full_clears_threshold"]]["feature"]
        .sort_values()
        .tolist()
    ),
    "full_population_moneyline_squared_variance_signals": (
        verdict[verdict["ml_sq_full_clears_threshold"]]["feature"]
        .sort_values()
        .tolist()
    ),
    "artifacts_written": {
        "verdict": str(verdict_path),
        "summary": str(summary_path),
        "all_signal": str(all_signal_path),
        "trajectory_signal": str(trajectory_path),
        "yoy_stability": str(yoy_path),
    },
}

with open(summary_path, "w") as f:
    json.dump(summary_payload, f, indent=2)

print("Artifacts written:")
print(f"- {verdict_path}")
print(f"- {summary_path}")
print(f"- {all_signal_path}")
print(f"- {trajectory_path}")
print(f"- {yoy_path}")

print("\nSummary payload:")
print(json.dumps(summary_payload, indent=2))

assert verdict_path.exists(), "ERROR: Verdict CSV was not written."
assert summary_path.exists(), "ERROR: Summary JSON was not written."
assert all_signal_path.exists(), "ERROR: all_signal CSV was not written."
assert trajectory_path.exists(), "ERROR: trajectory CSV was not written."
assert yoy_path.exists(), "ERROR: YoY CSV was not written."

print("\nCell 10 complete.")

Artifacts written:
- artifacts/style_tempo_verdict.csv
- artifacts/style_tempo_summary.json
- artifacts/style_tempo_all_signal.csv
- artifacts/style_tempo_trajectory_signal.csv
- artifacts/style_tempo_yoy_stability.csv

Summary payload:
{
  "notebook": "eda_09_style_tempo_delta.ipynb",
  "status": "rebuilt_clean",
  "seasons": [
    2022,
    2023,
    2024
  ],
  "holdout_season_excluded": 2025,
  "base_valid_games": 1607,
  "analysis_games": 1604,
  "dropped_games_missing_required_controls": 3,
  "style_dimensions_tested": 24,
  "style_delta_features_tested": 24,
  "partial_r_threshold": 0.08,
  "primary_controls": [
    "close_game_epa_delta",
    "sp_rating_delta"
  ],
  "outcomes_tested": [
    "spread",
    "over_under",
    "moneyline_abs_margin_variance",
    "moneyline_squared_margin_variance"
  ],
  "methodology": {
    "grain": "one row per game after home-minus-away style delta construction",
    "raw_metric_grain": "one row per team per game from raw.plays",
    "style_met

In [42]:
# Cell 11 — Final notebook summary and handoff notes

print("=" * 80)
print("EDA 09 — Style and Tempo Delta Analysis — FINAL SUMMARY")
print("=" * 80)

print("\nPopulation:")
print(f"- Valid FBS conference games: {len(valid_games):,}")
print(f"- Analysis games after required-control drop: {len(style_delta_analysis):,}")
print(f"- Dropped games: {len(valid_games) - len(style_delta_analysis):,}")
print("- Seasons: 2022, 2023, 2024")
print("- Holdout season excluded: 2025")

print("\nMethod:")
print("- raw.plays aggregated to one row per team per game")
print("- game table built as home-minus-away style deltas")
print("- primary controls: close_game_epa_delta, sp_rating_delta")
print("- partial-r threshold: 0.08")
print("- stratification: Full, P4, G5, each individual conference")
print("- trajectory buckets: conference game 1, 2–4, 5–8, 9–12")
print("- YoY stability used only for prior-seed eligibility")

print("\nCritical interpretation rule:")
print(
    "These are in-game style metrics. They qualify whether the football dimension "
    "matters. They do not by themselves prove pregame deployability. Pregame rolling "
    "or current-season versions must be tested before model inclusion."
)

print("\nFinal verdict counts:")
print(verdict["final_verdict"].value_counts().to_string())

print("\nRecommended deployment roles:")
print(verdict["recommended_deployment_role"].value_counts().to_string())

print("\nFull-population spread signals:")
spread_full_features = (
    verdict[verdict["spread_full_clears_threshold"]]
    .sort_values("spread_full_abs_partial_r", ascending=False)
    [["feature", "spread_full_partial_r", "spread_full_abs_partial_r"]]
)
display(spread_full_features)

print("\nFull-population over/under signals:")
ou_full_features = (
    verdict[verdict["ou_full_clears_threshold"]]
    .sort_values("ou_full_abs_partial_r", ascending=False)
    [["feature", "ou_full_partial_r", "ou_full_abs_partial_r"]]
)
display(ou_full_features)

print("\nFull-population moneyline abs variance signals:")
ml_abs_full_features = (
    verdict[verdict["ml_abs_full_clears_threshold"]]
    .sort_values("ml_abs_full_abs_partial_r", ascending=False)
    [["feature", "ml_abs_full_partial_r", "ml_abs_full_abs_partial_r"]]
)
display(ml_abs_full_features)

print("\nFull-population moneyline squared variance signals:")
ml_sq_full_features = (
    verdict[verdict["ml_sq_full_clears_threshold"]]
    .sort_values("ml_sq_full_abs_partial_r", ascending=False)
    [["feature", "ml_sq_full_partial_r", "ml_sq_full_abs_partial_r"]]
)
display(ml_sq_full_features)

print("\nPrior-seed eligible style features:")
prior_seed_features = (
    verdict[verdict["stable_for_prior_seed"]]
    .sort_values("avg_yoy_r", ascending=False)
    [["feature", "avg_yoy_r", "prior_seed_verdict"]]
)
display(prior_seed_features)

print("\nTop candidate rows:")
display(
    verdict[
        verdict["recommended_deployment_role"].isin([
            "test_prior_and_pregame_rolling_version",
            "test_pregame_rolling_current_season_version",
        ])
    ]
    .sort_values(
        [
            "recommended_deployment_role",
            "spread_full_abs_partial_r",
            "ml_abs_full_abs_partial_r",
            "ou_full_abs_partial_r",
        ],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

print("\nLocked findings to carry forward:")
print(
    """
1. Style/tempo dimensions are mostly game-level signals, not strong prior seeds.
2. rush_rate_std_downs and rush_rate_pass_downs are the only weak prior-seed candidates.
3. rush_rate_std_downs_delta is the strongest broad spread signal and holds across the season arc.
4. Pass-game dimensions matter for spread, especially:
   - off_success_rate_pass
   - def_success_rate_pass_allowed
   - off_epa_pass
   - def_epa_pass_allowed
5. Full-population over/under signal is weak, but bucket/conference-specific O/U signals exist.
6. Moneyline variance signal is most clearly tied to sack-rate mismatch and explosiveness mismatch.
7. No feature should be promoted directly from in-game diagnostic form to production form.
   Next step is to test pregame-deployable rolling/current-season versions.
"""
)

print("\nArtifacts:")
print(f"- {verdict_path}")
print(f"- {summary_path}")
print(f"- {all_signal_path}")
print(f"- {trajectory_path}")
print(f"- {yoy_path}")

print("\nNext notebook dependency:")
print(
    "eda_10_style_archetypes.ipynb can proceed, but it should use the rebuilt "
    "style_tempo_verdict.csv and must not use the invalidated previous version."
)

print("\nCell 11 complete.")
print("EDA 09 rebuild complete.")

EDA 09 — Style and Tempo Delta Analysis — FINAL SUMMARY

Population:
- Valid FBS conference games: 1,607
- Analysis games after required-control drop: 1,604
- Dropped games: 3
- Seasons: 2022, 2023, 2024
- Holdout season excluded: 2025

Method:
- raw.plays aggregated to one row per team per game
- game table built as home-minus-away style deltas
- primary controls: close_game_epa_delta, sp_rating_delta
- partial-r threshold: 0.08
- stratification: Full, P4, G5, each individual conference
- trajectory buckets: conference game 1, 2–4, 5–8, 9–12
- YoY stability used only for prior-seed eligibility

Critical interpretation rule:
These are in-game style metrics. They qualify whether the football dimension matters. They do not by themselves prove pregame deployability. Pregame rolling or current-season versions must be tested before model inclusion.

Final verdict counts:
final_verdict
candidate_game_level_signal_only                   22
candidate_game_level_signal_and_weak_prior_seed     2

,feature,spread_full_partial_r,spread_full_abs_partial_r
11,rush_rate_std_downs,0.292691,0.292691
12,rush_rate_pass_downs,0.216574,0.216574
10,off_pts_per_opportunity,0.146895,0.146895
22,def_pts_per_opportunity_allowed,-0.146895,0.146895
1,off_success_rate_pass,0.142680,0.142680
15,def_success_rate_pass_allowed,-0.142680,0.142680
5,off_epa_pass,0.088056,0.088056
18,def_epa_pass_allowed,-0.088056,0.088056



Full-population over/under signals:


,feature,ou_full_partial_r,ou_full_abs_partial_r



Full-population moneyline abs variance signals:


...


Prior-seed eligible style features:


,feature,avg_yoy_r,prior_seed_verdict
11,rush_rate_std_downs,0.489047,weak_prior_seed_candidate
12,rush_rate_pass_downs,0.464781,weak_prior_seed_candidate



Top candidate rows:


,feature,feature_delta,final_verdict,recommended_deployment_role,spread_verdict,spread_full_partial_r,spread_full_abs_partial_r,spread_full_clears_threshold,spread_p4_partial_r,spread_p4_clears_threshold,...,moneyline_trajectory_verdict,ml_abs_trajectory_buckets_clearing,ml_abs_trajectory_buckets_clearing_list,ml_sq_trajectory_buckets_clearing,ml_sq_trajectory_buckets_clearing_list,prior_seed_verdict,avg_yoy_r,stable_for_prior_seed,strongly_stable_for_prior_seed,methodology_note
0,off_pts_per_opportunity,off_pts_per_opportunity_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.146895,0.146895,True,0.177533,True,...,bucket_specific,0,,2,"conf_game_1, conf_games_9_12",not_prior_seed,0.170339,False,False,Signal tested using in-game home-minus-away st...
1,def_pts_per_opportunity_allowed,def_pts_per_opportunity_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.146895,0.146895,True,-0.177533,True,...,bucket_specific,0,,2,"conf_game_1, conf_games_9_12",not_prior_seed,0.217245,False,False,Signal tested using in-game home-minus-away st...
2,off_success_rate_pass,off_success_rate_pass_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.142680,0.142680,True,0.163745,True,...,bucket_specific,1,conf_games_9_12,1,conf_games_9_12,not_prior_seed,0.295336,False,False,Signal tested using in-game home-minus-away st...
3,def_success_rate_pass_allowed,def_success_rate_pass_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.142680,0.142680,True,-0.163745,True,...,bucket_specific,1,conf_games_9_12,1,conf_games_9_12,not_prior_seed,0.272995,False,False,Signal tested using in-game home-minus-away st...
4,off_epa_pass,off_epa_pass_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,0.088056,0.088056,True,0.073747,False,...,no_trajectory_signal,0,,0,,not_prior_seed,0.275521,False,False,Signal tested using in-game home-minus-away st...
5,def_epa_pass_allowed,def_epa_pass_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,full_population_signal,-0.088056,0.088056,True,-0.073747,False,...,no_trajectory_signal,0,,0,,not_prior_seed,0.297431,False,False,Signal tested using in-game home-minus-away st...
6,off_success_rate_pass_downs,off_success_rate_pass_downs_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,0.075526,0.075526,False,0.064213,False,...,holds_or_recurs_across_multiple_buckets,2,"conf_game_1, conf_games_5_8",1,conf_game_1,not_prior_seed,0.253089,False,False,Signal tested using in-game home-minus-away st...
7,off_sack_rate_allowed,off_sack_rate_allowed_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,-0.071307,0.071307,False,-0.043892,False,...,holds_or_recurs_across_multiple_buckets,3,"conf_game_1, conf_games_2_4, conf_games_9_12",3,"conf_game_1, conf_games_2_4, conf_games_9_12",not_prior_seed,0.243776,False,False,Signal tested using in-game home-minus-away st...
8,def_sack_rate,def_sack_rate_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,0.071307,0.071307,False,0.043892,False,...,holds_or_recurs_across_multiple_buckets,3,"conf_game_1, conf_games_2_4, conf_games_9_12",3,"conf_game_1, conf_games_2_4, conf_games_9_12",not_prior_seed,0.198099,False,False,Signal tested using in-game home-minus-away st...
9,off_explosive_rate_10,off_explosive_rate_10_delta,candidate_game_level_signal_only,test_pregame_rolling_current_season_version,tier_specific_signal,-0.071171,0.071171,False,-0.118043,True,...,holds_or_recurs_across_multiple_buckets,1,conf_games_9_12,2,"conf_game_1, conf_games_9_12",not_prior_seed,0.327651,False,False,Signal tested using in-game home-m


Locked findings to carry forward:

1. Style/tempo dimensions are mostly game-level signals, not strong prior seeds.
2. rush_rate_std_downs and rush_rate_pass_downs are the only weak prior-seed candidates.
3. rush_rate_std_downs_delta is the strongest broad spread signal and holds across the season arc.
4. Pass-game dimensions matter for spread, especially:
   - off_success_rate_pass
   - def_success_rate_pass_allowed
   - off_epa_pass
   - def_epa_pass_allowed
5. Full-population over/under signal is weak, but bucket/conference-specific O/U signals exist.
6. Moneyline variance signal is most clearly tied to sack-rate mismatch and explosiveness mismatch.
7. No feature should be promoted directly from in-game diagnostic form to production form.
   Next step is to test pregame-deployable rolling/current-season versions.


Artifacts:
- artifacts/style_tempo_verdict.csv
- artifacts/style_tempo_summary.json
- artifacts/style_tempo_all_signal.csv
- artifacts/style_tempo_trajectory_signal.csv
